# 02 — RNN & LSTM Flickr8k Image Captioning

**Dataset**: Flickr8k (~8.000 gambar, 5 caption per gambar)
**Feature Extractor**: InceptionV3 pretrained (dim=2048)
**Decoder**: SimpleRNN dan LSTM dengan arsitektur pre-inject

---

## Cara Pakai

> **Jalankan semua cell dari atas ke bawah secara berurutan.**

### Alur Kerja
```
[Setup]    gpu-setup → colab-mount → colab-clone → fix-patches → path-setup
              ↓
[Bagian 1] Preprocessing Flickr8k (parse captions, split, vocab)
              ↓
[Bagian 2] Ekstraksi CNN Features (InceptionV3, cached)
              ↓
[Bagian 3] Persiapan Array Training (teacher forcing)
              ↓
[Bagian 4] Training 6 variasi RNN (layer 1/2/3 × hidden 128/512)
              ↓
[Bagian 5] Training 6 variasi LSTM
              ↓
[Bagian 6] Evaluasi & Perbandingan
           ├── BLEU Keras RNN & LSTM (test set)
           ├── Scratch RNN & LSTM inference (load Keras weights)
           ├── Tabel perbandingan BLEU-1/2/3/4
           ├── Contoh kualitatif (GT vs model)
           └── Training curves + BLEU comparison plot
```

### Hasil Disimpan ke Drive
- Bobot RNN  → `MyDrive/weights/rnn/*.h5`
- Bobot LSTM → `MyDrive/weights/lstm/*.h5`
- Hasil JSON → `MyDrive/results/rnn_lstm/*.json`
- Plot       → `MyDrive/results/rnn_lstm/*.png`

In [7]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'[GPU] Ditemukan {len(gpus)} GPU: {[g.name for g in gpus]}')
    print(f'[TF]  Versi TensorFlow : {tf.__version__}')
    print(f'[TF]  Built with CUDA  : {tf.test.is_built_with_cuda()}')
else:
    print('[GPU] Tidak ada GPU terdeteksi — menggunakan CPU.')
    try:
        import google.colab
        print('      Di Colab: Runtime → Change runtime type → GPU (T4/L4), lalu Restart session.')
    except ImportError:
        print('      Di lokal: pastikan driver NVIDIA + tensorflow[and-cuda] sudah terinstall.')
    print(f'[TF]  Versi TensorFlow : {tf.__version__}')
    print(f'[TF]  Built with CUDA  : {tf.test.is_built_with_cuda()}')


[GPU] Tidak ada GPU terdeteksi — menggunakan CPU.
      Di lokal: pastikan driver NVIDIA + tensorflow[and-cuda] sudah terinstall.
[TF]  Versi TensorFlow : 2.10.0
[TF]  Built with CUDA  : True


In [8]:
import sys

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('[Colab] Google Drive terpasang.')
else:
    print('[Lokal] Tidak di Colab — skip Drive mount.')


[Lokal] Tidak di Colab — skip Drive mount.


In [9]:
import sys, os

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

REPO_URL = 'https://github.com/danenftyessir/ChosaHeidan_Tubes-2_IF3270.git'
REPO_DIR = '/content/ChosaHeidan_Tubes-2_IF3270'

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {REPO_URL} {REPO_DIR}')
    else:
        os.system(f'git -C {REPO_DIR} fetch origin')
        os.system(f'git -C {REPO_DIR} reset --hard origin/main')
    print(f'[Repo] Kode tersedia di {REPO_DIR}/src/')
else:
    print('[Lokal] Skip clone.')


[Lokal] Skip clone.


In [10]:
import sys

# 'model_keras' dan 'train' di-import sebagai bare name dari rnn/keras/ dan lstm/keras/
_PREFIXES = ('rnn', 'lstm', 'shared', 'caption_preprocess', 'model_keras', 'train')

_removed = [
    k for k in list(sys.modules.keys())
    if any(k == p or k.startswith(p + '.') for p in _PREFIXES)
]
for k in _removed:
    del sys.modules[k]

print(f'[Reload] {len(_removed)} modul dihapus dari cache:')
for k in sorted(_removed):
    print(f'  - {k}')
if not _removed:
    print('  (tidak ada modul yang di-cache — aman dilanjutkan)')


[Reload] 6 modul dihapus dari cache:
  - caption_preprocess
  - lstm
  - lstm.bonus
  - lstm.bonus.bonus_init_inject
  - lstm.scratch
  - lstm.scratch.model_scratch


In [11]:
import os, re

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    REPO_DIR = '/content/ChosaHeidan_Tubes-2_IF3270'
    _src = os.path.join(REPO_DIR, 'src')

    # 1. Buat __init__.py di semua package dir
    for _d in ['', 'shared', 'cnn', 'cnn/scratch', 'cnn/keras', 'cnn/utils', 'cnn/bonus',
               'lstm', 'lstm/scratch', 'lstm/keras', 'lstm/bonus',
               'rnn', 'rnn/scratch', 'rnn/keras', 'rnn/bonus']:
        _init = os.path.join(_src, _d, '__init__.py')
        if not os.path.exists(_init):
            open(_init, 'w').close()

    def _fix_rel(path, bare_import, make_block):
        with open(path, 'r') as f:
            c = f.read()
        c = re.sub(
            r'(?m)^([ ]*)try:[ ]*\n[ ]*' + re.escape(bare_import.lstrip()) +
            r'[ ]*\n[ ]*except ImportError:[ ]*\n[^\n]*',
            lambda m: m.group(1) + bare_import.lstrip(), c
        )
        c = re.sub(
            r'^([ ]*)' + re.escape(bare_import.lstrip()) + r'$',
            lambda m: make_block(m.group(1)), c, flags=re.MULTILINE
        )
        with open(path, 'w') as f:
            f.write(c)

    # 2. Fix shared/dense.py (dipakai oleh embedding & scratch models)
    _fix_rel(
        os.path.join(_src, 'shared', 'dense.py'),
        'from .activations import get_activation',
        lambda ind: (f'{ind}try:\n'
                     f'{ind}    from .activations import get_activation\n'
                     f'{ind}except ImportError:\n'
                     f'{ind}    from activations import get_activation')
    )

    print('[Fix-02] Patches applied — RNN/LSTM imports OK')
else:
    print('[Fix-02] Lokal — skip patch')


[Fix-02] Lokal — skip patch


In [12]:
import os, sys
import numpy as np

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

DRIVE_FLICKR_ID = '11TSyjWcx3mnp6lCXogtNnHrmSyxH5Mew'

if IN_COLAB:
    REPO_DIR = '/content/ChosaHeidan_Tubes-2_IF3270'
    SRC_DIR  = os.path.join(REPO_DIR, 'src')
    _MYDRIVE = '/content/drive/MyDrive'

    FLICKR8K_DIR     = os.path.join(_MYDRIVE, 'flickr8k')
    RNN_WEIGHTS_DIR  = os.path.join(_MYDRIVE, 'weights', 'rnn')
    LSTM_WEIGHTS_DIR = os.path.join(_MYDRIVE, 'weights', 'lstm')
    VOCAB_DIR        = os.path.join(_MYDRIVE, 'vocab')
    FEATURES_DIR     = os.path.join(_MYDRIVE, 'features')
    RESULTS_DIR      = os.path.join(_MYDRIVE, 'results', 'rnn_lstm')

    # Fallback: Drive API jika folder belum ter-mount
    if not os.path.exists(FLICKR8K_DIR):
        print('[Path] Folder flickr8k tidak ditemukan — pakai Drive API...')
        from google.colab import auth
        auth.authenticate_user()
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
        import io, concurrent.futures

        _svc = build('drive', 'v3')

        def _dl_folder(folder_id, dest, workers=6):
            os.makedirs(dest, exist_ok=True)
            items, page_token = [], None
            while True:
                resp = _svc.files().list(
                    q=f"'{folder_id}' in parents and trashed=false",
                    fields='nextPageToken,files(id,name,mimeType)',
                    pageToken=page_token, pageSize=1000
                ).execute()
                items.extend(resp.get('files', []))
                page_token = resp.get('nextPageToken')
                if not page_token:
                    break
            files = [(i, os.path.join(dest, i['name'])) for i in items
                     if i['mimeType'] != 'application/vnd.google-apps.folder']
            dirs  = [(i, os.path.join(dest, i['name'])) for i in items
                     if i['mimeType'] == 'application/vnd.google-apps.folder']
            def _one(pair):
                item, path = pair
                if os.path.exists(path):
                    return
                req = _svc.files().get_media(fileId=item['id'])
                with io.FileIO(path, 'wb') as fh:
                    dl = MediaIoBaseDownload(fh, req, chunksize=8*1024*1024)
                    done = False
                    while not done:
                        _, done = dl.next_chunk()
            with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as ex:
                list(ex.map(_one, files))
            for sub, sub_dest in dirs:
                _dl_folder(sub['id'], sub_dest, workers)

        print('[Drive API] Mengunduh flickr8k ...')
        _dl_folder(DRIVE_FLICKR_ID, FLICKR8K_DIR)

else:
    SRC_DIR = os.path.abspath('.')
    if not os.path.exists(os.path.join(SRC_DIR, 'rnn')):
        SRC_DIR = os.path.join(os.path.abspath('.'), 'src')
    PROJECT_ROOT     = os.path.dirname(SRC_DIR)
    FLICKR8K_DIR     = os.path.join(PROJECT_ROOT, 'data', 'flickr8k')
    RNN_WEIGHTS_DIR  = os.path.join(PROJECT_ROOT, 'weights', 'rnn')
    LSTM_WEIGHTS_DIR = os.path.join(PROJECT_ROOT, 'weights', 'lstm')
    VOCAB_DIR        = os.path.join(PROJECT_ROOT, 'data', 'vocab')
    FEATURES_DIR     = os.path.join(PROJECT_ROOT, 'data', 'features')
    RESULTS_DIR      = os.path.join(PROJECT_ROOT, 'results', 'rnn_lstm')

sys.path.insert(0, SRC_DIR)
sys.path.insert(0, os.path.join(SRC_DIR, 'shared'))

for d in [RNN_WEIGHTS_DIR, LSTM_WEIGHTS_DIR, VOCAB_DIR, FEATURES_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'[Path] SRC_DIR          : {SRC_DIR}  (exists: {os.path.exists(SRC_DIR)})')
print(f'[Path] FLICKR8K_DIR     : {FLICKR8K_DIR}  (exists: {os.path.exists(FLICKR8K_DIR)})')
print(f'[Path] RNN_WEIGHTS_DIR  : {RNN_WEIGHTS_DIR}')
print(f'[Path] LSTM_WEIGHTS_DIR : {LSTM_WEIGHTS_DIR}')
print(f'[Path] VOCAB_DIR        : {VOCAB_DIR}')
print(f'[Path] FEATURES_DIR     : {FEATURES_DIR}')

if IN_COLAB:
    assert os.path.exists(os.path.join(FLICKR8K_DIR, 'Images')),         f'flickr8k/Images/ tidak ada di: {FLICKR8K_DIR}'
    assert os.path.exists(os.path.join(FLICKR8K_DIR, 'captions.txt')),         f'flickr8k/captions.txt tidak ada di: {FLICKR8K_DIR}'
    print('[Path] Semua path OK')


[Path] SRC_DIR          : c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\src  (exists: True)
[Path] FLICKR8K_DIR     : c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\data\flickr8k  (exists: True)
[Path] RNN_WEIGHTS_DIR  : c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\weights\rnn
[Path] LSTM_WEIGHTS_DIR : c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\weights\lstm
[Path] VOCAB_DIR        : c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\data\vocab
[Path] FEATURES_DIR     : c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran

---

## Bagian 1 — Preprocessing Dataset Flickr8k

Langkah persiapan sebelum training RNN/LSTM:
1. Parse `captions.txt` → dictionary `{image_id: [5 captions]}`
2. Buat train/val/test split (80/10/10) → simpan ke Drive
3. Bangun vocabulary dari training captions (kata dengan freq ≥ 5)

> Hasil disimpan ke `VOCAB_DIR` sehingga tidak perlu diulang di sesi berikutnya.

In [13]:
import json
import numpy as np
from shared.caption_preprocess import (
    split_captions_file, build_vocabulary,
    save_vocabulary, load_vocabulary,
    get_max_caption_length
)

# ── 1. Parse captions.txt ─────────────────────────────────────────────────────
captions_path = os.path.join(FLICKR8K_DIR, 'captions.txt')
print(f'[B5] Membaca captions: {captions_path}')
captions_dict = split_captions_file(captions_path)
all_image_ids = sorted(captions_dict.keys())
print(f'[B5] Gambar: {len(all_image_ids)} | Caption per gambar: 5')

# ── 2. Buat atau muat train/val/test split ────────────────────────────────────
train_ids_path = os.path.join(VOCAB_DIR, 'train_ids.txt')
val_ids_path   = os.path.join(VOCAB_DIR, 'val_ids.txt')
test_ids_path  = os.path.join(VOCAB_DIR, 'test_ids.txt')

def _load_ids(p):
    return [l.strip() for l in open(p, encoding='utf-8') if l.strip()]

# Muat split HANYA jika file ada DAN tidak kosong
_splits_valid = (
    os.path.exists(train_ids_path) and os.path.getsize(train_ids_path) > 0 and
    os.path.exists(val_ids_path)   and os.path.getsize(val_ids_path)   > 0 and
    os.path.exists(test_ids_path)  and os.path.getsize(test_ids_path)  > 0
)

if _splits_valid:
    train_ids = _load_ids(train_ids_path)
    val_ids   = _load_ids(val_ids_path)
    test_ids  = _load_ids(test_ids_path)
    # Validasi: semua ID harus ada di captions_dict
    _known = set(all_image_ids)
    train_ids = [i for i in train_ids if i in _known]
    val_ids   = [i for i in val_ids   if i in _known]
    test_ids  = [i for i in test_ids  if i in _known]
    if len(train_ids) == 0:
        _splits_valid = False  # Corrupt → recreate

if _splits_valid:
    print(f'[B5] Split dimuat  — train:{len(train_ids)}, val:{len(val_ids)}, test:{len(test_ids)}')
else:
    print('[B5] Split file kosong/tidak valid — membuat split baru...')
    np.random.seed(42)
    perm    = np.random.permutation(len(all_image_ids))
    n_train = int(len(all_image_ids) * 0.80)
    n_val   = int(len(all_image_ids) * 0.10)
    train_ids = [all_image_ids[i] for i in perm[:n_train]]
    val_ids   = [all_image_ids[i] for i in perm[n_train:n_train+n_val]]
    test_ids  = [all_image_ids[i] for i in perm[n_train+n_val:]]
    for path, ids in [(train_ids_path, train_ids),
                      (val_ids_path,   val_ids),
                      (test_ids_path,  test_ids)]:
        with open(path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(ids))
    print(f'[B5] Split dibuat  — train:{len(train_ids)}, val:{len(val_ids)}, test:{len(test_ids)}')

# ── 3. Bangun atau muat vocabulary ────────────────────────────────────────────
vocab_path    = os.path.join(VOCAB_DIR, 'word2idx.json')
idx2word_path = os.path.join(VOCAB_DIR, 'idx2word.json')

if os.path.exists(vocab_path) and os.path.getsize(vocab_path) > 0:
    word2idx = load_vocabulary(vocab_path)
    with open(idx2word_path, encoding='utf-8') as f:
        idx2word = {int(k): v for k, v in json.load(f).items()}
    print(f'[B5] Vocab dimuat  — {len(word2idx)} kata')
else:
    train_caps = {img: captions_dict[img] for img in train_ids if img in captions_dict}
    word2idx, idx2word, vocab_size_actual = build_vocabulary(captions_dict, min_freq=5)
    save_vocabulary(word2idx, vocab_path)
    with open(idx2word_path, 'w', encoding='utf-8') as f:
        json.dump({str(k): v for k, v in idx2word.items()}, f)
    print(f'[B5] Vocab dibangun — {len(word2idx)} kata (min_freq=5)')

VOCAB_SIZE = len(word2idx)
MAX_LENGTH = min(get_max_caption_length(captions_dict) + 2, 40)  # +2 start/end

print(f'\n[B5] VOCAB_SIZE = {VOCAB_SIZE}')
print(f'[B5] MAX_LENGTH = {MAX_LENGTH}  (termasuk <start> dan <end>)')


[B5] Membaca captions: c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\data\flickr8k\captions.txt
[B5] Gambar: 8091 | Caption per gambar: 5
[B5] Split file kosong/tidak valid — membuat split baru...
[B5] Split dibuat  — train:6472, val:809, test:810
Vocabulary disimpan ke: c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\data\vocab\word2idx.json (2975 kata)
[B5] Vocab dibangun — 2975 kata (min_freq=5)

[B5] VOCAB_SIZE = 2975
[B5] MAX_LENGTH = 35  (termasuk <start> dan <end>)


## Bagian 2 — Ekstraksi CNN Features (InceptionV3)

Ekstrak feature vector (dim=2048) dari seluruh gambar Flickr8k menggunakan InceptionV3 pretrained.

> **Hanya berjalan sekali** — hasil disimpan ke `FEATURES_DIR/flickr8k_inception.npy`.
> Sesi berikutnya langsung load dari file. Estimasi waktu ekstraksi: ~10–15 menit di T4/L4 GPU.

In [8]:
import json, os, sys

# Guard: pastikan SRC_DIR ada di sys.path sebelum import shared
if not any(os.path.isdir(os.path.join(p, 'shared')) for p in sys.path):
    for _cand in [
        '/content/ChosaHeidan_Tubes-2_IF3270/src',
        os.path.join(os.path.abspath('.'), 'src'),
        os.path.abspath('.'),
    ]:
        if os.path.isdir(os.path.join(_cand, 'shared')):
            sys.path.insert(0, _cand)
            print(f'[B6] sys.path patched → {_cand}')
            break

from shared.feature_extract import (
    extract_features_inceptionv3, save_cnn_features, load_cnn_features
)

FEATURES_PATH  = os.path.join(FEATURES_DIR, 'flickr8k_inception.npy')
FEAT_IDS_PATH  = os.path.join(FEATURES_DIR, 'flickr8k_image_ids.json')
IMAGE_DIR      = os.path.join(FLICKR8K_DIR, 'Images')

if os.path.exists(FEATURES_PATH) and os.path.exists(FEAT_IDS_PATH):
    print('[B6] Features sudah ada — skip ekstraksi, langsung muat...')
    raw_features   = load_cnn_features(FEATURES_PATH)
    with open(FEAT_IDS_PATH, encoding='utf-8') as f:
        feat_image_ids = json.load(f)
else:
    # ── Recover all_image_ids (3 lapis fallback) ──────────────────────────────
    if 'all_image_ids' not in globals() or len(all_image_ids) == 0:
        # Lapis 1: coba muat dari split files
        _split_files = [
            os.path.join(VOCAB_DIR, 'train_ids.txt'),
            os.path.join(VOCAB_DIR, 'val_ids.txt'),
            os.path.join(VOCAB_DIR, 'test_ids.txt'),
        ]
        if all(os.path.exists(p) for p in _split_files):
            all_image_ids = []
            for _p in _split_files:
                all_image_ids += [l.strip() for l in open(_p, encoding='utf-8') if l.strip()]

        # Lapis 2: split files kosong/tidak ada → parse langsung captions.txt
        if len(all_image_ids) == 0:
            _captions_txt = os.path.join(FLICKR8K_DIR, 'captions.txt')
            if os.path.exists(_captions_txt):
                from shared.caption_preprocess import split_captions_file
                _caps = split_captions_file(_captions_txt)
                all_image_ids = sorted(_caps.keys())
                print(f'[B6] all_image_ids di-recover dari captions.txt: {len(all_image_ids)} gambar')
            else:
                raise RuntimeError(
                    f'captions.txt tidak ditemukan di: {FLICKR8K_DIR}\n'
                    'Pastikan struktur Drive: flickr8k/captions.txt dan flickr8k/Images/'
                )

        if len(all_image_ids) == 0:
            raise RuntimeError('all_image_ids masih kosong setelah semua fallback. Cek isi captions.txt.')

        print(f'[B6] all_image_ids tersedia: {len(all_image_ids)} gambar')

    print(f'[B6] Ekstraksi InceptionV3 untuk {len(all_image_ids)} gambar...')
    print(f'     Image dir : {IMAGE_DIR}')
    print(f'     Output    : {FEATURES_PATH}')
    raw_features   = extract_features_inceptionv3(
        all_image_ids, image_dir=IMAGE_DIR, batch_size=32, verbose=True
    )
    feat_image_ids = all_image_ids
    save_cnn_features(raw_features, FEATURES_PATH)
    with open(FEAT_IDS_PATH, 'w', encoding='utf-8') as f:
        json.dump(feat_image_ids, f)

# Buat lookup dict: image_id -> feature vector
features_dict = {img_id: raw_features[i]
                 for i, img_id in enumerate(feat_image_ids)}
FEATURE_DIM   = raw_features.shape[1]

print(f'\n[B6] features_dict : {len(features_dict)} gambar')
print(f'[B6] FEATURE_DIM   : {FEATURE_DIM}')


[B6] Features sudah ada — skip ekstraksi, langsung muat...
[Feature Extraction] Features dimuat dari: /mnt/c/Users/user/ChosaHeidan_Tubes-2_IF3270/data/features/flickr8k_inception.npy (shape: (8091, 2048))

[B6] features_dict : 8091 gambar
[B6] FEATURE_DIM   : 2048


## Bagian 3 — Persiapan Array Training (Teacher Forcing)

Konversi caption teks → array numerik untuk training RNN/LSTM.

**Format teacher forcing:**
```
caption : [<start>, w1, w2, w3, <end>, <pad>, ...]
input   : [<start>, w1, w2, w3, <end>, <pad>]   ← seq_train
target  : [w1,      w2, w3, <end>, <pad>, <pad>] ← lbl_train
```
Setiap gambar punya 5 caption → tiap gambar menghasilkan 5 baris training.

In [9]:
# Unpack jika load_vocabulary mengembalikan tuple
if isinstance(word2idx, tuple):
    word2idx, idx2word = word2idx
    print(f'[Fix] word2idx: {len(word2idx)} kata')

[Fix] word2idx: 2653 kata


In [10]:
from shared.caption_preprocess import prepare_training_data
import numpy as np

def build_split_arrays(image_ids, captions_dict, features_dict, word2idx, max_length):
    """
    Buat (cnn_features, input_seqs, target_seqs, img_ids) untuk satu split.
    Setiap gambar × 5 caption = N baris array.
    Returns img_ids agar bisa dipakai untuk qualitative display.
    """
    matched_ids, full_seqs = prepare_training_data(
        captions_dict, image_ids, word2idx,
        max_length=max_length, add_start=True, add_end=True
    )
    valid = [(i, img_id) for i, img_id in enumerate(matched_ids)
             if img_id in features_dict]
    if not valid:
        raise ValueError('Tidak ada gambar yang cocok dengan features_dict!')

    idxs    = [v[0] for v in valid]
    img_ids = [v[1] for v in valid]

    seqs      = full_seqs[idxs]
    cnn_feats = np.array([features_dict[i] for i in img_ids])

    input_seqs  = seqs[:, :-1]
    target_seqs = seqs[:, 1:]

    return cnn_feats, input_seqs, target_seqs, img_ids

print('[B3] Membangun array training (teacher forcing)...')
cnn_train, seq_train, lbl_train, train_img_ids = build_split_arrays(
    train_ids, captions_dict, features_dict, word2idx, MAX_LENGTH)
cnn_val,   seq_val,   lbl_val,   val_img_ids   = build_split_arrays(
    val_ids,   captions_dict, features_dict, word2idx, MAX_LENGTH)
cnn_test,  seq_test,  lbl_test,  test_img_ids  = build_split_arrays(
    test_ids,  captions_dict, features_dict, word2idx, MAX_LENGTH)

SEQ_LENGTH = seq_train.shape[1]   # MAX_LENGTH - 1
IMAGE_DIR  = os.path.join(FLICKR8K_DIR, 'Images')

print(f'\n{"Split":<8} | {"Pasang":>8} | {"cnn_shape":>15} | {"seq_shape":>15}')
print('-' * 55)
for name, cnn, seq in [('Train', cnn_train, seq_train),
                        ('Val',   cnn_val,   seq_val),
                        ('Test',  cnn_test,  seq_test)]:
    print(f'{name:<8} | {cnn.shape[0]:>8,} | {str(cnn.shape):>15} | {str(seq.shape):>15}')

print(f'\n[B3] SEQ_LENGTH = {SEQ_LENGTH}')
print(f'[B3] VOCAB_SIZE = {VOCAB_SIZE}, FEATURE_DIM = {FEATURE_DIM}')
print('[B3] Siap untuk training RNN & LSTM!')


[B3] Membangun array training (teacher forcing)...

Split    |   Pasang |       cnn_shape |       seq_shape
-------------------------------------------------------
Train    |   32,360 |   (32360, 2048) |     (32360, 34)
Val      |    4,045 |    (4045, 2048) |      (4045, 34)
Test     |    4,050 |    (4050, 2048) |      (4050, 34)

[B3] SEQ_LENGTH = 34
[B3] VOCAB_SIZE = 2, FEATURE_DIM = 2048
[B3] Siap untuk training RNN & LSTM!


---

## Bagian 4 — Training RNN

Train **6 variasi RNN** (SimpleRNN decoder, arsitektur preinject):

| Hyperparameter | Nilai |
|---|---|
| Jumlah layer | 1, 2, 3 |
| Hidden dim | 128, 512 |

**Total**: 3 × 2 = **6 model RNN**

> Bobot terbaik (best val_loss) setiap model disimpan ke `weights/rnn/`.
> Early stopping patience = 5. Estimasi waktu: ~20–40 menit di L4 GPU.

In [12]:
# 1. Pastikan word2idx sudah dict, bukan tuple
if isinstance(word2idx, tuple):
    word2idx, idx2word = word2idx

# 2. Hitung ulang VOCAB_SIZE
VOCAB_SIZE = len(word2idx)
print(f'VOCAB_SIZE: {VOCAB_SIZE}')  # harus 2653


VOCAB_SIZE: 2653


In [17]:
import json as _json, os as _os
from rnn.keras.train import train_with_variations as rnn_train_variations

_rnn_cmp_json = _os.path.join(RESULTS_DIR, 'rnn_comparison.json')

if _os.path.exists(_rnn_cmp_json):
    with open(_rnn_cmp_json) as _f:
        rnn_comparison = _json.load(_f)
    rnn_results = None
    print(f'[B8] Cache ditemukan — skip training, {len(rnn_comparison)} model dimuat dari JSON.')
else:
    print('[B8] Mulai training 6 variasi RNN...')
    print(f'     cnn_train : {cnn_train.shape}')
    print(f'     seq_train : {seq_train.shape}')
    print(f'     VOCAB_SIZE: {VOCAB_SIZE}, FEATURE_DIM: {FEATURE_DIM}')
    print(f'     SEQ_LENGTH: {SEQ_LENGTH}')
    rnn_results = rnn_train_variations(
        cnn_features_train=cnn_train,
        train_seq=seq_train,
        train_labels=lbl_train,
        val_cnn_features=cnn_val,
        val_seq=seq_val,
        val_labels=lbl_val,
        vocab_size=VOCAB_SIZE,
        embed_dim=256,
        layer_variations=[1, 2, 3],
        hidden_variations=[128, 512],
        feature_dim=FEATURE_DIM,
        seq_max_length=SEQ_LENGTH,
        epochs=30,
        batch_size=64,
        lr=0.001,
        weights_dir=RNN_WEIGHTS_DIR,
        architecture='preinject',
        dropout=0.3,
        verbose=1,
    )
    rnn_comparison = None
    print(f'[B8] Selesai — {len(rnn_results)} model RNN dilatih.')


[B8] Mulai training 6 variasi RNN...
     cnn_train : (32360, 2048)
     seq_train : (32360, 34)
     VOCAB_SIZE: 2653, FEATURE_DIM: 2048
     SEQ_LENGTH: 34

[Training Variations] preinject
  Layer variations: [1, 2, 3]
  Hidden variations: [128, 512]
  Total models: 6

[Training] rnn_l1_h128_preinject
  embed_dim=256, hidden_dim=128, num_layers=1, epochs=30, batch_size=64
Epoch 1/30


W0000 00:00:1778722471.020707   15253 assert_op.cc:39] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
I0000 00:00:1778722471.775061   15253 dot_merger.cc:481] Merging Dots in computation: rnn_preinject_1_rnn_decoder_1_while_body_73671_grad_74720_const_0__.157.clone.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1778722471.775405   15253 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_75381__.163
I0000 00:00:1778722473.367762   17801 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_34', 8380 bytes spill stores, 8468 bytes spill loads

I0000 00:00:1778722474.578059   17788 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_67', 160 bytes spill stores, 156 bytes spill loads



504/506 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6582 - loss: 2.6957

W0000 00:00:1778722510.490855   15252 assert_op.cc:39] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
I0000 00:00:1778722511.207734   15252 dot_merger.cc:481] Merging Dots in computation: rnn_preinject_1_rnn_decoder_1_while_body_73671_grad_74720_const_0__.157.clone.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1778722511.208071   15252 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_75381__.163
I0000 00:00:1778722513.397928   17847 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_67', 120 bytes spill stores, 120 bytes spill loads

I0000 00:00:1778722514.538205   17848 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_34', 8580 bytes spill stores, 8672 bytes spill loads



506/506 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.6584 - loss: 2.6924

...

505/506 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.6706 - loss: 2.2462

I0000 00:00:1778725613.111021   15253 dot_merger.cc:481] Merging Dots in computation: rnn_preinject_1_rnn_layer_2_1_while_body_447351_grad_448401_const_0__.167.clone.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1778725613.111376   15253 dot_merger.cc:481] Merging Dots in computation: rnn_preinject_1_rnn_layer_1_1_while_body_447177_grad_448699_const_0__.174.clone.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1778725613.111571   15253 dot_merger.cc:481] Merging Dots in computation: rnn_preinject_1_rnn_layer_0_1_while_body_447003_grad_448997_const_0__.181.clone.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1778725613.111937   15253 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_449868__.187


506/506 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.6707 - loss: 2.2453

I0000 00:00:1778725632.294918   15253 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_453057__.93
I0000 00:00:1778725641.856711   15253 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_453057__.93


506/506 ━━━━━━━━━━━━━━━━━━━━ 94s 143ms/step - accuracy: 0.7131 - loss: 1.7679 - val_accuracy: 0.7464 - val_loss: 1.4018
Epoch 2/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 33s 64ms/step - accuracy: 0.7500 - loss: 1.3769 - val_accuracy: 0.7564 - val_loss: 1.2768
Epoch 3/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 32s 64ms/step - accuracy: 0.7576 - loss: 1.2816 - val_accuracy: 0.7613 - val_loss: 1.2157
Epoch 4/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 32s 62ms/step - accuracy: 0.7607 - loss: 1.2421 - val_accuracy: 0.7629 - val_loss: 1.2142
Epoch 5/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 31s 62ms/step - accuracy: 0.7619 - loss: 1.2225 - val_accuracy: 0.7676 - val_loss: 1.1707
Epoch 6/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 31s 62ms/step - accuracy: 0.7638 - loss: 1.2016 - val_accuracy: 0.7692 - val_loss: 1.1614
Epoch 7/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 32s 63ms/step - accuracy: 0.7636 - loss: 1.2026 - val_accuracy: 0.7693 - val_loss: 1.1542
Epoch 8/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 32s 64ms/step - accuracy: 0.7657 - loss: 1.1778 - val_accura

In [17]:
import json as _json, os as _os, re as _re
import tensorflow as tf
from rnn.keras.model_keras import build_rnn_decoder_preinject
from rnn.keras.train import compare_training_results as rnn_compare

# Reconstruct path variables jika setup cell belum dijalankan
if 'RESULTS_DIR' not in globals():
    _src = _os.path.abspath('.')
    if not _os.path.exists(_os.path.join(_src, 'rnn')):
        _src = _os.path.join(_os.path.abspath('.'), 'src')
    _project_root = _os.path.dirname(_src)
    RESULTS_DIR     = _os.path.join(_project_root, 'results', 'rnn_lstm')
    RNN_WEIGHTS_DIR = _os.path.join(_project_root, 'weights', 'rnn')
    _os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f'[B8-rk] RESULTS_DIR reconstruct -> {RESULTS_DIR}')

# Fallback: load dari cache jika nb02-b4 belum dijalankan di sesi ini
if 'rnn_comparison' not in globals():
    _rnn_cmp_json = _os.path.join(RESULTS_DIR, 'rnn_comparison.json')
    if _os.path.exists(_rnn_cmp_json):
        with open(_rnn_cmp_json) as _f:
            rnn_comparison = _json.load(_f)
        rnn_results = None
        print(f'[B8-rk] Fallback: {len(rnn_comparison)} model dimuat dari cache JSON.')
    else:
        _hist_files = [
            f for f in _os.listdir(RNN_WEIGHTS_DIR)
            if f.endswith('_history.json')
        ] if _os.path.isdir(RNN_WEIGHTS_DIR) else []
        if _hist_files:
            rnn_comparison = {}
            for _hf in sorted(_hist_files):
                _name = _hf.replace('_history.json', '')
                with open(_os.path.join(RNN_WEIGHTS_DIR, _hf)) as _f:
                    _hist = _json.load(_f)
                _vl = _hist.get('val_loss', [])
                rnn_comparison[_name] = {
                    'best_val_loss': float(min(_vl)) if _vl else None,
                    'epochs_trained': len(_hist.get('loss', [])),
                }
            rnn_results = None
            with open(_rnn_cmp_json, 'w') as _f:
                _json.dump(rnn_comparison, _f, indent=2)
            print(f'[B8-rk] Rekonstruksi dari {len(_hist_files)} history file -> disimpan ke cache.')
        else:
            raise RuntimeError(
                'rnn_comparison tidak ditemukan, cache JSON tidak ada, '
                'dan tidak ada *_history.json di weights/rnn/ -- '
                'jalankan cell nb02-b4 terlebih dahulu.'
            )

if rnn_comparison is None:
    rnn_comparison = rnn_compare(
        rnn_results,
        save_path=_os.path.join(RESULTS_DIR, 'rnn_comparison.json'),
    )

rnn_ranked = sorted(
    [(k, v) for k, v in rnn_comparison.items()
     if v.get('best_val_loss') is not None],
    key=lambda x: x[1]['best_val_loss'],
)

print(f'\n{"="*65}')
print('  RANKING RNN -- Best Val Loss (lower is better)')
print(f'{"="*65}')
for i, (name, data) in enumerate(rnn_ranked, 1):
    mark = ' <-- BEST' if i == 1 else ''
    print(
        f'  {i:2d}. {name:<35} '
        f'val_loss={data["best_val_loss"]:.4f}  '
        f'ep={data["epochs_trained"]}{mark}'
    )


def _load_keras3_weights(model, path):
    import h5py as _h5
    with _h5.File(path, 'r') as _f:
        if 'layers' not in _f:
            raise ValueError(f'Unknown .weights.h5 format in {path}')
        _lg = _f['layers']
        _cnt = {}
        for _layer in model.layers:
            _cls = type(_layer).__name__.lower()
            _n = _cnt.get(_cls, 0)
            _cnt[_cls] = _n + 1
            _key = _cls if _n == 0 else f'{_cls}_{_n}'
            if _key not in _lg:
                continue
            _grp = _lg[_key]
            # Direct layer vars (Dense, Embedding, ...)
            if 'vars' in _grp and len(_grp['vars']) > 0:
                for _i, _v in enumerate(_layer.variables):
                    if str(_i) in _grp['vars']:
                        _v.assign(_grp['vars'][str(_i)][()])
            # RNN/SimpleRNN cell vars
            if 'cell' in _grp and hasattr(_layer, 'cell'):
                _cgrp = _grp['cell']
                if 'vars' in _cgrp:
                    for _i, _v in enumerate(_layer.cell.variables):
                        if str(_i) in _cgrp['vars']:
                            _v.assign(_cgrp['vars'][str(_i)][()])
            # TimeDistributed inner-layer vars
            if 'layer' in _grp and hasattr(_layer, 'layer'):
                _igrp = _grp['layer']
                if 'vars' in _igrp:
                    for _i, _v in enumerate(_layer.layer.variables):
                        if str(_i) in _igrp['vars']:
                            _v.assign(_igrp['vars'][str(_i)][()])


if rnn_ranked:
    BEST_RNN_NAME = rnn_ranked[0][0]

    if 'rnn_results' in globals() and rnn_results is not None and rnn_results.get(BEST_RNN_NAME, {}).get('model') is not None:
        BEST_RNN_MODEL = rnn_results[BEST_RNN_NAME]['model']
    else:
        _m = _re.match(r'rnn_l(\d+)_h(\d+)_preinject', BEST_RNN_NAME)
        _num_layers = int(_m.group(1))
        _hidden_dim = int(_m.group(2))
        _vocab_size  = globals().get('VOCAB_SIZE',  2653)
        _feature_dim = globals().get('FEATURE_DIM', 2048)
        _seq_length  = globals().get('SEQ_LENGTH',  34)

        _best_h5  = _os.path.join(RNN_WEIGHTS_DIR, f'{BEST_RNN_NAME}_best.h5')
        _best_wh5 = _os.path.join(RNN_WEIGHTS_DIR, f'{BEST_RNN_NAME}_best.weights.h5')

        BEST_RNN_MODEL = build_rnn_decoder_preinject(
            vocab_size=_vocab_size, embed_dim=256, hidden_dim=_hidden_dim,
            num_layers=_num_layers, feature_dim=_feature_dim,
            seq_max_length=_seq_length, dropout=0.3,
        )
        BEST_RNN_MODEL([
            tf.zeros((1, _feature_dim)),
            tf.zeros((1, _seq_length), dtype=tf.int32),
        ])

        if _os.path.exists(_best_h5):
            BEST_RNN_MODEL.load_weights(_best_h5)
            print(f'[B8] Bobot dimuat dari: {_best_h5}')
        elif _os.path.exists(_best_wh5):
            _load_keras3_weights(BEST_RNN_MODEL, _best_wh5)
            print(f'[B8] Bobot dimuat dari: {_best_wh5}')
        else:
            raise FileNotFoundError(f'Tidak ditemukan: {_best_h5} maupun {_best_wh5}')

    print(f'\n[B8] Best RNN model : {BEST_RNN_NAME}')
    print(f'[B8] Bobot tersimpan: {RNN_WEIGHTS_DIR}/{BEST_RNN_NAME}_best.weights.h5')



  RANKING RNN -- Best Val Loss (lower is better)
   1. rnn_l1_h512_preinject               val_loss=1.0285  ep=22 <-- BEST
   2. rnn_l1_h128_preinject               val_loss=1.0448  ep=30
   3. rnn_l2_h512_preinject               val_loss=1.0496  ep=30
   4. rnn_l2_h128_preinject               val_loss=1.0607  ep=30
   5. rnn_l3_h512_preinject               val_loss=1.0954  ep=30
   6. rnn_l3_h128_preinject               val_loss=1.0997  ep=30
[B8] Bobot dimuat dari: c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\weights\rnn\rnn_l1_h512_preinject_best.weights.h5

[B8] Best RNN model : rnn_l1_h512_preinject
[B8] Bobot tersimpan: c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\weights\rnn/rnn_l1_h512_preinject_best.weights.h5


### Bagian 4-Bonus — Pre-Inject vs Init-Inject Architecture (RNN)

Dua arsitektur injeksi CNN feature ke dalam decoder RNN:
- **Pre-Inject**: feature CNN dikonversi ke embed_dim dan menjadi token pertama (x_{-1})
- **Init-Inject**: RNN memproses token saja, lalu [h_T ; CNN_projected] → Dense → softmax

> Perbandingan menggunakan scratch model dengan bobot acak (demonstrasi arsitektur).

In [23]:
import sys
for k in list(sys.modules.keys()):
    if 'rnn' in k or 'bonus' in k:
        del sys.modules[k]

In [1]:
# Bonus — Init-Inject RNN: Training Keras + BLEU-4 Comparison
from rnn.bonus.bonus_init_inject import (
    RNNInitInject, build_initinject_from_config, compare_preinject_vs_initinject,
    train_rnn_initinject_keras, evaluate_bleu_rnn_initinject,
    load_initinject_weights_to_scratch,
)
from rnn.keras.model_keras import build_rnn_model
from rnn.scratch.model_scratch import RNNScratch
import json as _json, os as _os, re as _re, sys as _sys
import numpy as _np

# ---------- Variable fallbacks ----------
if "FEATURE_DIM" not in globals(): FEATURE_DIM = 2048
if "VOCAB_SIZE" not in globals(): VOCAB_SIZE = 2653
if "SEQ_LENGTH" not in globals(): SEQ_LENGTH = 34
if "rnn_results" not in globals(): rnn_results = None

if "BEST_RNN_NAME" not in globals():
    _proj_root = _os.path.dirname(_os.path.abspath("."))
    _cmp_json = _os.path.join(_proj_root, "results", "rnn_lstm", "rnn_comparison.json")
    with open(_cmp_json) as _f:
        _cmp = _json.load(_f)
    _ranked = sorted(
        [(k, v) for k, v in _cmp.items() if v.get("best_val_loss") is not None],
        key=lambda x: x[1]["best_val_loss"],
    )
    BEST_RNN_NAME = _ranked[0][0]
    print(f"[Bonus] BEST_RNN_NAME: {BEST_RNN_NAME}")

if "RNN_WEIGHTS_DIR" not in globals():
    RNN_WEIGHTS_DIR = _os.path.join(_os.path.dirname(_os.path.abspath(".")), "weights", "rnn")

if rnn_results is not None:
    best_rnn_cfg = rnn_results[BEST_RNN_NAME]["config"]
else:
    _m = _re.match(r"rnn_l(\d+)_h(\d+)_preinject", BEST_RNN_NAME)
    best_rnn_cfg = {
        "num_layers": int(_m.group(1)),
        "hidden_dim": int(_m.group(2)),
        "embed_dim": 256,
        "feature_dim": FEATURE_DIM,
    }

# ---------- Load vocabulary & test features ----------
if "idx2word" not in globals():
    _idx2word_path = _os.path.join(_os.path.dirname(_os.path.abspath(".")), "data", "idx2word.json")
    with open(_idx2word_path, encoding="utf-8") as _f:
        idx2word = {int(k): v for k, v in _json.load(_f).items()}
    print(f"[Bonus] idx2word dimuat: {len(idx2word)} kata")

if "cnn_test" not in globals():
    _proj = _os.path.dirname(_os.path.abspath("."))
    _feat = _os.path.join(_proj, "data", "flickr8k_inception.npy")
    _ids  = _os.path.join(_proj, "data", "flickr8k_image_ids.json")
    _test = _os.path.join(_proj, "data", "test_ids.txt")
    if _os.path.exists(_feat) and _os.path.exists(_ids) and _os.path.exists(_test):
        _raw = _np.load(_feat)
        with open(_ids) as _f:
            _all_ids = _json.load(_f)
        with open(_test) as _f:
            _test_ids = [l.strip() for l in _f if l.strip()]
        _id2idx = {img_id: i for i, img_id in enumerate(_all_ids)}
        cnn_test = _np.array([_raw[_id2idx[i]] for i in _test_ids if i in _id2idx])
        print(f"[Bonus] cnn_test: {cnn_test.shape}")
    else:
        print("[WARN] cnn_test tidak bisa direkonstruksi dari cache.")

if "gt_captions_test" not in globals():
    if "lbl_test" in globals() and "idx2word" in globals():
        _END_WORD = "<end>"
        _SKIP = {"<start>", "<end>", "<pad>", ""}
        gt_captions_test = []
        for _seq in lbl_test:
            _words = []
            for _t in _seq:
                _w = idx2word.get(int(_t), "")
                if _w == _END_WORD:
                    break
                if _w not in _SKIP:
                    _words.append(_w)
            gt_captions_test.append(" ".join(_words))
        print(f"[Bonus] gt_captions_test rekonstruksi: {len(gt_captions_test)} caption")
    else:
        # Rekonstruksi penuh
        _proj = _os.path.dirname(_os.path.abspath("."))
        if not any(_os.path.isdir(_os.path.join(p, "shared")) for p in _sys.path):
            for _cand in [_os.path.join(_proj, "src"), _proj]:
                if _os.path.isdir(_os.path.join(_cand, "shared")):
                    _sys.path.insert(0, _cand)
                    break
        from shared.caption_preprocess import split_captions_file, prepare_training_data
        _w2i_path = _os.path.join(_proj, "data", "word2idx.json")
        with open(_w2i_path, encoding="utf-8") as _f:
            _word2idx = _json.load(_f)
        _test_id_path = _os.path.join(_proj, "data", "test_ids.txt")
        with open(_test_id_path) as _f:
            _test_ids_full = [l.strip() for l in _f if l.strip()]
        _cap_dict = split_captions_file(_os.path.join(_proj, "data", "flickr8k", "captions.txt"))
        _feat_np = _np.load(_os.path.join(_proj, "data", "flickr8k_inception.npy"))
        with open(_os.path.join(_proj, "data", "flickr8k_image_ids.json")) as _f:
            _feat_ids = _json.load(_f)
        _features_dict = {img_id: _feat_np[i] for i, img_id in enumerate(_feat_ids)}
        _matched_ids, _full_seqs = prepare_training_data(
            _cap_dict, _test_ids_full, _word2idx, max_length=35, add_start=True, add_end=True)
        _valid = [(i, img_id) for i, img_id in enumerate(_matched_ids) if img_id in _features_dict]
        _idxs = [v[0] for v in _valid]
        _test_img_ids = [v[1] for v in _valid]
        _seqs = _full_seqs[_idxs]
        cnn_test = _np.array([_features_dict[i] for i in _test_img_ids])
        _lbl_test = _seqs[:, 1:]
        _END_WORD = "<end>"
        _SKIP = {"<start>", "<end>", "<pad>", ""}
        gt_captions_test = []
        for _seq in _lbl_test:
            _words = []
            for _t in _seq:
                _w = idx2word.get(int(_t), "")
                if _w == _END_WORD:
                    break
                if _w not in _SKIP:
                    _words.append(_w)
            gt_captions_test.append(" ".join(_words))
        print(f"[Bonus] gt_captions_test direkonstruksi penuh: {len(gt_captions_test)} caption")

# ---------- Build scratch models ----------
rnn_initinject = build_initinject_from_config(
    vocab_size=VOCAB_SIZE,
    embed_dim=best_rnn_cfg["embed_dim"],
    hidden_dim=best_rnn_cfg["hidden_dim"],
    num_layers=best_rnn_cfg["num_layers"],
    feature_dim=FEATURE_DIM,
)
rnn_initinject.build()
print(f"RNNInitInject dibangun: hidden_dim={best_rnn_cfg['hidden_dim']}, num_layers={best_rnn_cfg['num_layers']}")

if "rnn_scratch" not in globals():
    print("[Bonus] rnn_scratch belum ada — membangun dari bobot tersimpan...")
    rnn_scratch = RNNScratch(
        vocab_size=VOCAB_SIZE,
        embed_dim=best_rnn_cfg["embed_dim"],
        hidden_dim=best_rnn_cfg["hidden_dim"],
        num_layers=best_rnn_cfg["num_layers"],
        feature_dim=FEATURE_DIM,
    )
    rnn_scratch.build()
    _best_wh5_pre = _os.path.join(RNN_WEIGHTS_DIR, f"{BEST_RNN_NAME}_best.weights.h5")
    if _os.path.exists(_best_wh5_pre):
        rnn_scratch.load_weights_from_h5(_best_wh5_pre)
        print(f"  Bobot dimuat: {_best_wh5_pre}")
    else:
        print(f"  [WARN] Bobot tidak ditemukan: {_best_wh5_pre}")

# ---------- Load/Train Keras Init-Inject ----------
import tensorflow as _tf
_ii_keras_name = f"rnn_l{best_rnn_cfg['num_layers']}_h{best_rnn_cfg['hidden_dim']}_initinject"
_ii_wh5_best = _os.path.join(RNN_WEIGHTS_DIR, f"{_ii_keras_name}_best.weights.h5")
_ii_wh5      = _os.path.join(RNN_WEIGHTS_DIR, f"{_ii_keras_name}.weights.h5")

def _build_rnn_initinject_keras():
    _model = build_rnn_model(
        vocab_size=VOCAB_SIZE,
        embed_dim=best_rnn_cfg["embed_dim"],
        hidden_dim=best_rnn_cfg["hidden_dim"],
        num_layers=best_rnn_cfg["num_layers"],
        feature_dim=FEATURE_DIM,
        seq_max_length=SEQ_LENGTH,
        architecture="initinject_train",
        dropout=0.3,
    )
    _model([_tf.zeros((1, FEATURE_DIM)), _tf.zeros((1, SEQ_LENGTH), dtype=_tf.int32)], training=False)
    return _model

if _os.path.exists(_ii_wh5_best) or _os.path.exists(_ii_wh5):
    _load_path = _ii_wh5_best if _os.path.exists(_ii_wh5_best) else _ii_wh5
    print(f"[Bonus] Memuat Keras RNN Init-Inject dari: {_os.path.basename(_load_path)}")
    rnn_keras_initinject = _build_rnn_initinject_keras()
    rnn_keras_initinject.load_weights(_load_path)
    print("  Bobot Keras dimuat.")
else:
    print(f"[Bonus] Belum ada bobot Keras Init-Inject — melatih sekarang...")
    # Rekonstruksi training data jika perlu
    if not all(v in globals() for v in ["cnn_train", "seq_train", "lbl_train",
                                         "cnn_val", "seq_val", "lbl_val"]):
        print("  Merekonstruksi data latih/validasi...")
        _proj = _os.path.dirname(_os.path.abspath("."))
        if not any(_os.path.isdir(_os.path.join(p, "shared")) for p in _sys.path):
            for _cand in [_os.path.join(_proj, "src"), _proj]:
                if _os.path.isdir(_os.path.join(_cand, "shared")):
                    _sys.path.insert(0, _cand)
                    break
        from shared.caption_preprocess import split_captions_file, prepare_training_data
        if "word2idx" not in globals():
            with open(_os.path.join(_proj, "data", "word2idx.json"), encoding="utf-8") as _f:
                word2idx = _json.load(_f)
        if "captions_dict" not in globals():
            captions_dict = split_captions_file(_os.path.join(_proj, "data", "flickr8k", "captions.txt"))
        if "features_dict" not in globals():
            _feat_np2 = _np.load(_os.path.join(_proj, "data", "flickr8k_inception.npy"))
            with open(_os.path.join(_proj, "data", "flickr8k_image_ids.json")) as _f:
                _feat_ids2 = _json.load(_f)
            features_dict = {img_id: _feat_np2[i] for i, img_id in enumerate(_feat_ids2)}
        if "MAX_LENGTH" not in globals():
            MAX_LENGTH = 35
        for _split_name, _split_file, _split_cnn_var, _split_seq_var, _split_lbl_var in [
            ("train", "train_ids.txt", "cnn_train", "seq_train", "lbl_train"),
            ("val",   "val_ids.txt",   "cnn_val",   "seq_val",   "lbl_val"),
        ]:
            if _split_cnn_var not in globals():
                with open(_os.path.join(_proj, "data", _split_file)) as _f:
                    _split_ids = [l.strip() for l in _f if l.strip()]
                _m_ids, _seqs2 = prepare_training_data(
                    captions_dict, _split_ids, word2idx,
                    max_length=MAX_LENGTH, add_start=True, add_end=True)
                _valid2 = [(i, img_id) for i, img_id in enumerate(_m_ids) if img_id in features_dict]
                _idxs2 = [v[0] for v in _valid2]
                _split_cnn = _np.array([features_dict[v[1]] for v in _valid2])
                _split_seqs = _seqs2[_idxs2]
                globals()[_split_cnn_var] = _split_cnn
                globals()[_split_seq_var] = _split_seqs[:, :-1]
                globals()[_split_lbl_var] = _split_seqs[:, 1:]
                print(f"  {_split_name}: {_split_cnn.shape[0]} sample")
    rnn_keras_initinject, _ii_history, _ = train_rnn_initinject_keras(
        cnn_train, seq_train, lbl_train,
        cnn_val, seq_val, lbl_val,
        vocab_size=VOCAB_SIZE,
        cfg=best_rnn_cfg,
        weights_dir=RNN_WEIGHTS_DIR,
        epochs=30,
        batch_size=64,
    )
    _load_path = _ii_wh5_best if _os.path.exists(_ii_wh5_best) else _ii_wh5
    print(f"[Bonus] Training selesai. Bobot disimpan: {_os.path.basename(_load_path)}")

# Load bobot trained Keras ke scratch Init-Inject
if _os.path.exists(_load_path):
    load_initinject_weights_to_scratch(rnn_initinject, _load_path)
    print(f"[Bonus] Bobot dimuat ke RNNInitInject scratch.")

# ---------- Qualitative comparison ----------
print("\n[Bonus] Qualitative: Pre-Inject vs Init-Inject (5 sample):")
compare_preinject_vs_initinject(
    preinject_model=rnn_scratch,
    initinject_model=rnn_initinject,
    cnn_features=cnn_test[:5],
    idx2word=idx2word,
    max_length=SEQ_LENGTH,
    n_samples=5,
)

# ---------- BLEU-4 evaluation ----------
print("\n[Bonus] Evaluasi BLEU untuk RNN Init-Inject pada test set...")
_ii_metrics, _ii_preds = evaluate_bleu_rnn_initinject(
    rnn_keras_initinject, cnn_test, gt_captions_test, idx2word,
    max_length=SEQ_LENGTH, verbose=True,
)

# Load Pre-Inject BLEU (dari rnn_metrics.json atau dari globals)
if "rnn_metrics" in globals():
    _pre_metrics = rnn_metrics
else:
    _rnn_m_path = _os.path.join(_os.path.dirname(_os.path.abspath(".")), "results", "rnn_lstm", "rnn_metrics.json")
    if _os.path.exists(_rnn_m_path):
        with open(_rnn_m_path) as _f:
            _pre_metrics = _json.load(_f)
        print(f"[Bonus] Pre-Inject BLEU dimuat dari cache.")
    else:
        print("[WARN] rnn_metrics.json tidak ditemukan — Pre-Inject BLEU belum tersedia.")
        _pre_metrics = {}

# ---------- Comparison table ----------
print()
print("=" * 62)
print(f"{'RNN Pre-Inject vs Init-Inject — BLEU Comparison':^62}")
print("=" * 62)
print(f"{'Metric':<15} {'Pre-Inject':>15} {'Init-Inject':>15} {'Diff':>15}")
print("-" * 62)
for _key in ["bleu1", "bleu2", "bleu3", "bleu4", "meteor"]:
    _pre_val = _pre_metrics.get(_key, 0.0)
    _ii_val  = _ii_metrics.get(_key, 0.0)
    _diff    = _ii_val - _pre_val
    print(f"{_key.upper():<15} {_pre_val:>15.4f} {_ii_val:>15.4f} {_diff:>+15.4f}")
print("=" * 62)
rnn_initinject_metrics = _ii_metrics


[Bonus] BEST_RNN_NAME: rnn_l1_h512_preinject
[Bonus] idx2word dimuat: 2653 kata
[Bonus] cnn_test: (810, 2048)
[Bonus] gt_captions_test direkonstruksi penuh: 4050 caption
RNNInitInject dibangun: hidden_dim=512, num_layers=1
[Bonus] rnn_scratch belum ada — membangun dari bobot tersimpan...
Bobot berhasil dimuat dari: c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\weights\rnn\rnn_l1_h512_preinject_best.weights.h5
  Bobot dimuat: c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\weights\rnn\rnn_l1_h512_preinject_best.weights.h5
[Bonus] Belum ada bobot Keras Init-Inject — melatih sekarang...
  Merekonstruksi data latih/validasi...
  train: 32360 sample
  val: 4045 sample

[Training] rnn_l1_h512_initinject
  embed_dim=256, hidden_dim=512, num_layers=1, epochs=30, batch_size=64
Epoch 1/30
100/506 [====>.........................]

KeyboardInterrupt: 

---

## Bagian 5 — Training LSTM

Train **6 variasi LSTM** (arsitektur identik dengan RNN, LSTM cell menggantikan SimpleRNN):

| Hyperparameter | Nilai |
|---|---|
| Jumlah layer | 1, 2, 3 |
| Hidden dim | 128, 512 |

**Total**: 3 × 2 = **6 model LSTM**

> LSTM memiliki gating mechanism (forget, input, output gate) sehingga lebih baik
> menangani long-range dependencies dibanding SimpleRNN — namun lebih lambat saat training.
> Bobot terbaik disimpan ke `weights/lstm/`. Estimasi waktu: ~30–60 menit di L4 GPU.

In [13]:
import json as _json, os as _os
from lstm.keras.train import train_with_variations as lstm_train_variations

_lstm_cmp_json = _os.path.join(RESULTS_DIR, 'lstm_comparison.json')

if _os.path.exists(_lstm_cmp_json):
    with open(_lstm_cmp_json) as _f:
        lstm_comparison = _json.load(_f)
    lstm_results = None
    print(f'[B9] Cache ditemukan — skip training, {len(lstm_comparison)} model dimuat dari JSON.')
else:
    print('[B9] Mulai training 6 variasi LSTM...')
    lstm_results = lstm_train_variations(
        cnn_features_train=cnn_train,
        train_seq=seq_train,
        train_labels=lbl_train,
        val_cnn_features=cnn_val,
        val_seq=seq_val,
        val_labels=lbl_val,
        vocab_size=VOCAB_SIZE,
        embed_dim=256,
        layer_variations=[1, 2, 3],
        hidden_variations=[128, 512],
        feature_dim=FEATURE_DIM,
        seq_max_length=SEQ_LENGTH,
        epochs=30,
        batch_size=64,
        lr=0.001,
        weights_dir=LSTM_WEIGHTS_DIR,
        architecture='preinject',
        dropout=0.3,
        verbose=1,
    )
    lstm_comparison = None
    print(f'[B9] Selesai — {len(lstm_results)} model LSTM dilatih.')


[B9] Mulai training 6 variasi LSTM...

[Training Variations] preinject
  Layer variations: [1, 2, 3]
  Hidden variations: [128, 512]
  Total models: 6

[Training] lstm_l1_h128_preinject
  embed_dim=256, hidden_dim=128, num_layers=1, epochs=30, batch_size=64
Epoch 1/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 228s 442ms/step - accuracy: 0.7122 - loss: 1.8804 - val_accuracy: 0.7414 - val_loss: 1.4407
Epoch 2/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 223s 441ms/step - accuracy: 0.7507 - loss: 1.3828 - val_accuracy: 0.7571 - val_loss: 1.2910
Epoch 3/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 222s 440ms/step - accuracy: 0.7595 - loss: 1.2881 - val_accuracy: 0.7633 - val_loss: 1.2266
Epoch 4/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 222s 438ms/step - accuracy: 0.7637 - loss: 1.2370 - val_accuracy: 0.7671 - val_loss: 1.1860
Epoch 5/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 223s 440ms/step - accuracy: 0.7669 - loss: 1.2011 - val_accuracy: 0.7705 - val_loss: 1.1577
Epoch 6/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 221s 438ms/step - accuracy: 0.7690 - loss:

In [1]:
import json as _json, os as _os, re as _re
import tensorflow as tf
from lstm.keras.model_keras import build_lstm_decoder_preinject
from lstm.keras.train import compare_training_results as lstm_compare

# Reconstruct path variables jika setup cell belum dijalankan
if 'RESULTS_DIR' not in globals():
    _src = _os.path.abspath('.')
    if not _os.path.exists(_os.path.join(_src, 'rnn')):
        _src = _os.path.join(_os.path.abspath('.'), 'src')
    _project_root = _os.path.dirname(_src)
    RESULTS_DIR      = _os.path.join(_project_root, 'results', 'rnn_lstm')
    LSTM_WEIGHTS_DIR = _os.path.join(_project_root, 'weights', 'lstm')
    _os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f'[B9-rk] RESULTS_DIR reconstruct -> {RESULTS_DIR}')

# Fallback: load dari cache jika nb02-b5 belum dijalankan di sesi ini
if 'lstm_comparison' not in globals():
    _lstm_cmp_json = _os.path.join(RESULTS_DIR, 'lstm_comparison.json')
    if _os.path.exists(_lstm_cmp_json):
        with open(_lstm_cmp_json) as _f:
            lstm_comparison = _json.load(_f)
        lstm_results = None
        print(f'[B9-rk] Fallback: {len(lstm_comparison)} model dimuat dari cache JSON.')
    else:
        _hist_files = [
            f for f in _os.listdir(LSTM_WEIGHTS_DIR)
            if f.endswith('_history.json')
        ] if _os.path.isdir(LSTM_WEIGHTS_DIR) else []
        if _hist_files:
            lstm_comparison = {}
            for _hf in sorted(_hist_files):
                _name = _hf.replace('_history.json', '')
                with open(_os.path.join(LSTM_WEIGHTS_DIR, _hf)) as _f:
                    _hist = _json.load(_f)
                _vl = _hist.get('val_loss', [])
                lstm_comparison[_name] = {
                    'best_val_loss': float(min(_vl)) if _vl else None,
                    'epochs_trained': len(_hist.get('loss', [])),
                }
            lstm_results = None
            with open(_lstm_cmp_json, 'w') as _f:
                _json.dump(lstm_comparison, _f, indent=2)
            print(f'[B9-rk] Rekonstruksi dari {len(_hist_files)} history file -> disimpan ke cache.')
        else:
            raise RuntimeError(
                'lstm_comparison tidak ditemukan, cache JSON tidak ada, '
                'dan tidak ada *_history.json di weights/lstm/ -- '
                'jalankan cell nb02-b5 terlebih dahulu.'
            )

if lstm_comparison is None:
    lstm_comparison = lstm_compare(
        lstm_results,
        save_path=_os.path.join(RESULTS_DIR, 'lstm_comparison.json'),
    )

lstm_ranked = sorted(
    [(k, v) for k, v in lstm_comparison.items()
     if v.get('best_val_loss') is not None],
    key=lambda x: x[1]['best_val_loss'],
)

print(f'\n{"="*65}')
print('  RANKING LSTM -- Best Val Loss (lower is better)')
print(f'{"="*65}')
for i, (name, data) in enumerate(lstm_ranked, 1):
    mark = ' <-- BEST' if i == 1 else ''
    print(
        f'  {i:2d}. {name:<35} '
        f'val_loss={data["best_val_loss"]:.4f}  '
        f'ep={data["epochs_trained"]}{mark}'
    )


def _load_keras3_weights(model, path):
    import h5py as _h5
    with _h5.File(path, 'r') as _f:
        if 'layers' not in _f:
            raise ValueError(f'Unknown .weights.h5 format in {path}')
        _lg = _f['layers']
        _cnt = {}
        for _layer in model.layers:
            _cls = type(_layer).__name__.lower()
            _n = _cnt.get(_cls, 0)
            _cnt[_cls] = _n + 1
            _key = _cls if _n == 0 else f'{_cls}_{_n}'
            if _key not in _lg:
                continue
            _grp = _lg[_key]
            # Direct layer vars (Dense, Embedding, ...)
            if 'vars' in _grp and len(_grp['vars']) > 0:
                for _i, _v in enumerate(_layer.variables):
                    if str(_i) in _grp['vars']:
                        _v.assign(_grp['vars'][str(_i)][()])
            # LSTM cell vars
            if 'cell' in _grp and hasattr(_layer, 'cell'):
                _cgrp = _grp['cell']
                if 'vars' in _cgrp:
                    for _i, _v in enumerate(_layer.cell.variables):
                        if str(_i) in _cgrp['vars']:
                            _v.assign(_cgrp['vars'][str(_i)][()])
            # TimeDistributed inner-layer vars
            if 'layer' in _grp and hasattr(_layer, 'layer'):
                _igrp = _grp['layer']
                if 'vars' in _igrp:
                    for _i, _v in enumerate(_layer.layer.variables):
                        if str(_i) in _igrp['vars']:
                            _v.assign(_igrp['vars'][str(_i)][()])


if lstm_ranked:
    BEST_LSTM_NAME = lstm_ranked[0][0]

    if 'lstm_results' in globals() and lstm_results is not None and lstm_results.get(BEST_LSTM_NAME, {}).get('model') is not None:
        BEST_LSTM_MODEL = lstm_results[BEST_LSTM_NAME]['model']
    else:
        _m = _re.match(r'lstm_l(\d+)_h(\d+)_preinject', BEST_LSTM_NAME)
        _num_layers = int(_m.group(1))
        _hidden_dim = int(_m.group(2))
        _vocab_size  = globals().get('VOCAB_SIZE',  2653)
        _feature_dim = globals().get('FEATURE_DIM', 2048)
        _seq_length  = globals().get('SEQ_LENGTH',  34)

        _best_h5  = _os.path.join(LSTM_WEIGHTS_DIR, f'{BEST_LSTM_NAME}_best.h5')
        _best_wh5 = _os.path.join(LSTM_WEIGHTS_DIR, f'{BEST_LSTM_NAME}_best.weights.h5')

        BEST_LSTM_MODEL = build_lstm_decoder_preinject(
            vocab_size=_vocab_size, embed_dim=256, hidden_dim=_hidden_dim,
            num_layers=_num_layers, feature_dim=_feature_dim,
            seq_max_length=_seq_length, dropout=0.3,
        )
        BEST_LSTM_MODEL([
            tf.zeros((1, _feature_dim)),
            tf.zeros((1, _seq_length), dtype=tf.int32),
        ])

        if _os.path.exists(_best_h5):
            BEST_LSTM_MODEL.load_weights(_best_h5)
            print(f'[B9] Bobot dimuat dari: {_best_h5}')
        elif _os.path.exists(_best_wh5):
            _load_keras3_weights(BEST_LSTM_MODEL, _best_wh5)
            print(f'[B9] Bobot dimuat dari: {_best_wh5}')
        else:
            raise FileNotFoundError(f'Tidak ditemukan: {_best_h5} maupun {_best_wh5}')

    print(f'\n[B9] Best LSTM model : {BEST_LSTM_NAME}')
    print(f'[B9] Bobot tersimpan : {LSTM_WEIGHTS_DIR}/{BEST_LSTM_NAME}_best.weights.h5')


[B9-rk] RESULTS_DIR reconstruct -> c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\results\rnn_lstm
[B9-rk] Fallback: 6 model dimuat dari cache JSON.

  RANKING LSTM -- Best Val Loss (lower is better)
   1. lstm_l1_h512_preinject              val_loss=0.9427  ep=21 <-- BEST
   2. lstm_l2_h512_preinject              val_loss=0.9513  ep=24
   3. lstm_l3_h512_preinject              val_loss=0.9728  ep=29
   4. lstm_l2_h128_preinject              val_loss=0.9911  ep=30
   5. lstm_l1_h128_preinject              val_loss=0.9917  ep=30
   6. lstm_l3_h128_preinject              val_loss=1.0078  ep=30
[B9] Bobot dimuat dari: c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\weights\lstm\lstm_l1_h512_preinject_best.weights.h5

[B9] Best LSTM model : lstm_l1_h512_preinject
[B9] Bobot tersimpan : c:\Users\HYPE R Series\OneDrive - Inst

### Bagian 5-Bonus — Pre-Inject vs Init-Inject Architecture (LSTM)

Sama seperti RNN, LSTM juga mendukung dua arsitektur injeksi CNN feature.

In [ ]:
# nb02-b5-ii: Bonus — Init-Inject LSTM: Training Keras + BLEU-4 Comparison
from lstm.bonus.bonus_init_inject import (
    LSTMInitInject, build_lstm_initinject_from_config,
    compare_lstm_preinject_vs_initinject,
    transfer_preinject_weights_to_initinject,
    train_lstm_initinject_keras, evaluate_bleu_lstm_initinject,
    load_initinject_weights_to_scratch,
)
from lstm.keras.model_keras import build_lstm_model
from lstm.scratch.model_scratch import LSTMScratch
import json as _json, os as _os, re as _re, sys as _sys
import numpy as _np

# ---------- Variable fallbacks ----------
if "FEATURE_DIM" not in globals(): FEATURE_DIM = 2048
if "VOCAB_SIZE" not in globals(): VOCAB_SIZE = 2653
if "SEQ_LENGTH" not in globals(): SEQ_LENGTH = 34
if "lstm_results" not in globals(): lstm_results = None

if "BEST_LSTM_NAME" not in globals():
    _proj_root = _os.path.dirname(_os.path.abspath("."))
    _cmp_json = _os.path.join(_proj_root, "results", "rnn_lstm", "lstm_comparison.json")
    with open(_cmp_json) as _f:
        _cmp = _json.load(_f)
    _ranked = sorted(
        [(k, v) for k, v in _cmp.items() if v.get("best_val_loss") is not None],
        key=lambda x: x[1]["best_val_loss"],
    )
    BEST_LSTM_NAME = _ranked[0][0]
    print(f"[Bonus] BEST_LSTM_NAME: {BEST_LSTM_NAME}")

if "LSTM_WEIGHTS_DIR" not in globals():
    LSTM_WEIGHTS_DIR = _os.path.join(_os.path.dirname(_os.path.abspath(".")), "weights", "lstm")

if lstm_results is not None:
    best_lstm_cfg = lstm_results[BEST_LSTM_NAME]["config"]
else:
    _m = _re.match(r"lstm_l(\d+)_h(\d+)_preinject", BEST_LSTM_NAME)
    best_lstm_cfg = {
        "num_layers": int(_m.group(1)),
        "hidden_dim": int(_m.group(2)),
        "embed_dim": 256,
        "feature_dim": FEATURE_DIM,
    }

# ---------- Load vocabulary & test features ----------
if "idx2word" not in globals():
    _idx2word_path = _os.path.join(_os.path.dirname(_os.path.abspath(".")), "data", "idx2word.json")
    with open(_idx2word_path, encoding="utf-8") as _f:
        idx2word = {int(k): v for k, v in _json.load(_f).items()}
    print(f"[Bonus] idx2word dimuat: {len(idx2word)} kata")

if "cnn_test" not in globals():
    _proj = _os.path.dirname(_os.path.abspath("."))
    _feat = _os.path.join(_proj, "data", "flickr8k_inception.npy")
    _ids  = _os.path.join(_proj, "data", "flickr8k_image_ids.json")
    _test = _os.path.join(_proj, "data", "test_ids.txt")
    if _os.path.exists(_feat) and _os.path.exists(_ids) and _os.path.exists(_test):
        _raw = _np.load(_feat)
        with open(_ids) as _f:
            _all_ids = _json.load(_f)
        with open(_test) as _f:
            _test_ids = [l.strip() for l in _f if l.strip()]
        _id2idx = {img_id: i for i, img_id in enumerate(_all_ids)}
        cnn_test = _np.array([_raw[_id2idx[i]] for i in _test_ids if i in _id2idx])
        print(f"[Bonus] cnn_test: {cnn_test.shape}")
    else:
        print("[WARN] cnn_test tidak bisa direkonstruksi dari cache.")

if "gt_captions_test" not in globals():
    if "lbl_test" in globals() and "idx2word" in globals():
        _END_WORD = "<end>"
        _SKIP = {"<start>", "<end>", "<pad>", ""}
        gt_captions_test = []
        for _seq in lbl_test:
            _words = []
            for _t in _seq:
                _w = idx2word.get(int(_t), "")
                if _w == _END_WORD:
                    break
                if _w not in _SKIP:
                    _words.append(_w)
            gt_captions_test.append(" ".join(_words))
        print(f"[Bonus] gt_captions_test rekonstruksi: {len(gt_captions_test)} caption")
    else:
        _proj = _os.path.dirname(_os.path.abspath("."))
        if not any(_os.path.isdir(_os.path.join(p, "shared")) for p in _sys.path):
            for _cand in [_os.path.join(_proj, "src"), _proj]:
                if _os.path.isdir(_os.path.join(_cand, "shared")):
                    _sys.path.insert(0, _cand)
                    break
        from shared.caption_preprocess import split_captions_file, prepare_training_data
        _w2i_path = _os.path.join(_proj, "data", "word2idx.json")
        with open(_w2i_path, encoding="utf-8") as _f:
            _word2idx = _json.load(_f)
        _test_id_path = _os.path.join(_proj, "data", "test_ids.txt")
        with open(_test_id_path) as _f:
            _test_ids_full = [l.strip() for l in _f if l.strip()]
        _cap_dict = split_captions_file(_os.path.join(_proj, "data", "flickr8k", "captions.txt"))
        _feat_np = _np.load(_os.path.join(_proj, "data", "flickr8k_inception.npy"))
        with open(_os.path.join(_proj, "data", "flickr8k_image_ids.json")) as _f:
            _feat_ids = _json.load(_f)
        _features_dict = {img_id: _feat_np[i] for i, img_id in enumerate(_feat_ids)}
        _matched_ids, _full_seqs = prepare_training_data(
            _cap_dict, _test_ids_full, _word2idx, max_length=35, add_start=True, add_end=True)
        _valid = [(i, img_id) for i, img_id in enumerate(_matched_ids) if img_id in _features_dict]
        _idxs = [v[0] for v in _valid]
        _test_img_ids = [v[1] for v in _valid]
        _seqs = _full_seqs[_idxs]
        cnn_test = _np.array([_features_dict[i] for i in _test_img_ids])
        _lbl_test = _seqs[:, 1:]
        _END_WORD = "<end>"
        _SKIP = {"<start>", "<end>", "<pad>", ""}
        gt_captions_test = []
        for _seq in _lbl_test:
            _words = []
            for _t in _seq:
                _w = idx2word.get(int(_t), "")
                if _w == _END_WORD:
                    break
                if _w not in _SKIP:
                    _words.append(_w)
            gt_captions_test.append(" ".join(_words))
        print(f"[Bonus] gt_captions_test direkonstruksi penuh: {len(gt_captions_test)} caption")

# ---------- Build scratch models ----------
lstm_initinject = build_lstm_initinject_from_config(
    vocab_size=VOCAB_SIZE,
    embed_dim=best_lstm_cfg["embed_dim"],
    hidden_dim=best_lstm_cfg["hidden_dim"],
    num_layers=best_lstm_cfg["num_layers"],
    feature_dim=FEATURE_DIM,
)
print(f"LSTMInitInject dibangun: hidden_dim={best_lstm_cfg['hidden_dim']}, num_layers={best_lstm_cfg['num_layers']}")

if "lstm_scratch" not in globals():
    print("[Bonus] lstm_scratch belum ada — membangun dari bobot tersimpan...")
    lstm_scratch = LSTMScratch(
        vocab_size=VOCAB_SIZE,
        embed_dim=best_lstm_cfg["embed_dim"],
        hidden_dim=best_lstm_cfg["hidden_dim"],
        num_layers=best_lstm_cfg["num_layers"],
        feature_dim=FEATURE_DIM,
    )
    lstm_scratch.build()
    _best_h5_pre  = _os.path.join(LSTM_WEIGHTS_DIR, f"{BEST_LSTM_NAME}_best.h5")
    _best_wh5_pre = _os.path.join(LSTM_WEIGHTS_DIR, f"{BEST_LSTM_NAME}_best.weights.h5")
    if _os.path.exists(_best_h5_pre):
        lstm_scratch.load_weights_from_h5(_best_h5_pre)
        print(f"  Bobot dimuat: {_best_h5_pre}")
    elif _os.path.exists(_best_wh5_pre):
        lstm_scratch.load_weights_from_h5(_best_wh5_pre)
        print(f"  Bobot dimuat: {_best_wh5_pre}")
    else:
        print(f"  [WARN] Bobot tidak ditemukan.")

# Transfer pre-inject weights ke init-inject scratch
transfer_preinject_weights_to_initinject(lstm_scratch, lstm_initinject)

# ---------- Load/Train Keras Init-Inject ----------
import tensorflow as _tf
_ii_keras_name = f"lstm_l{best_lstm_cfg['num_layers']}_h{best_lstm_cfg['hidden_dim']}_initinject"
_ii_wh5_best = _os.path.join(LSTM_WEIGHTS_DIR, f"{_ii_keras_name}_best.weights.h5")
_ii_wh5      = _os.path.join(LSTM_WEIGHTS_DIR, f"{_ii_keras_name}.weights.h5")

def _build_lstm_initinject_keras():
    _model = build_lstm_model(
        vocab_size=VOCAB_SIZE,
        embed_dim=best_lstm_cfg["embed_dim"],
        hidden_dim=best_lstm_cfg["hidden_dim"],
        num_layers=best_lstm_cfg["num_layers"],
        feature_dim=FEATURE_DIM,
        seq_max_length=SEQ_LENGTH,
        architecture="initinject_train",
        dropout=0.3,
    )
    _model([_tf.zeros((1, FEATURE_DIM)), _tf.zeros((1, SEQ_LENGTH), dtype=_tf.int32)], training=False)
    return _model

if _os.path.exists(_ii_wh5_best) or _os.path.exists(_ii_wh5):
    _load_path = _ii_wh5_best if _os.path.exists(_ii_wh5_best) else _ii_wh5
    print(f"[Bonus] Memuat Keras LSTM Init-Inject dari: {_os.path.basename(_load_path)}")
    lstm_keras_initinject = _build_lstm_initinject_keras()
    lstm_keras_initinject.load_weights(_load_path)
    print("  Bobot Keras dimuat.")
else:
    print(f"[Bonus] Belum ada bobot Keras Init-Inject — melatih sekarang...")
    if not all(v in globals() for v in ["cnn_train", "seq_train", "lbl_train",
                                         "cnn_val", "seq_val", "lbl_val"]):
        print("  Merekonstruksi data latih/validasi...")
        _proj = _os.path.dirname(_os.path.abspath("."))
        if not any(_os.path.isdir(_os.path.join(p, "shared")) for p in _sys.path):
            for _cand in [_os.path.join(_proj, "src"), _proj]:
                if _os.path.isdir(_os.path.join(_cand, "shared")):
                    _sys.path.insert(0, _cand)
                    break
        from shared.caption_preprocess import split_captions_file, prepare_training_data
        if "word2idx" not in globals():
            with open(_os.path.join(_proj, "data", "word2idx.json"), encoding="utf-8") as _f:
                word2idx = _json.load(_f)
        if "captions_dict" not in globals():
            captions_dict = split_captions_file(_os.path.join(_proj, "data", "flickr8k", "captions.txt"))
        if "features_dict" not in globals():
            _feat_np2 = _np.load(_os.path.join(_proj, "data", "flickr8k_inception.npy"))
            with open(_os.path.join(_proj, "data", "flickr8k_image_ids.json")) as _f:
                _feat_ids2 = _json.load(_f)
            features_dict = {img_id: _feat_np2[i] for i, img_id in enumerate(_feat_ids2)}
        if "MAX_LENGTH" not in globals():
            MAX_LENGTH = 35
        for _split_name, _split_file, _split_cnn_var, _split_seq_var, _split_lbl_var in [
            ("train", "train_ids.txt", "cnn_train", "seq_train", "lbl_train"),
            ("val",   "val_ids.txt",   "cnn_val",   "seq_val",   "lbl_val"),
        ]:
            if _split_cnn_var not in globals():
                with open(_os.path.join(_proj, "data", _split_file)) as _f:
                    _split_ids = [l.strip() for l in _f if l.strip()]
                _m_ids, _seqs2 = prepare_training_data(
                    captions_dict, _split_ids, word2idx,
                    max_length=MAX_LENGTH, add_start=True, add_end=True)
                _valid2 = [(i, img_id) for i, img_id in enumerate(_m_ids) if img_id in features_dict]
                _idxs2 = [v[0] for v in _valid2]
                _split_cnn = _np.array([features_dict[v[1]] for v in _valid2])
                _split_seqs = _seqs2[_idxs2]
                globals()[_split_cnn_var] = _split_cnn
                globals()[_split_seq_var] = _split_seqs[:, :-1]
                globals()[_split_lbl_var] = _split_seqs[:, 1:]
                print(f"  {_split_name}: {_split_cnn.shape[0]} sample")
    lstm_keras_initinject, _ii_history, _ = train_lstm_initinject_keras(
        cnn_train, seq_train, lbl_train,
        cnn_val, seq_val, lbl_val,
        vocab_size=VOCAB_SIZE,
        cfg=best_lstm_cfg,
        weights_dir=LSTM_WEIGHTS_DIR,
        epochs=30,
        batch_size=64,
    )
    _load_path = _ii_wh5_best if _os.path.exists(_ii_wh5_best) else _ii_wh5
    print(f"[Bonus] Training selesai. Bobot: {_os.path.basename(_load_path)}")

# Load bobot trained Keras ke scratch Init-Inject
if _os.path.exists(_load_path):
    load_initinject_weights_to_scratch(lstm_initinject, _load_path)
    print(f"[Bonus] Bobot dimuat ke LSTMInitInject scratch.")

# ---------- Qualitative comparison ----------
print("\n[Bonus] Qualitative: Pre-Inject vs Init-Inject (5 sample):")
compare_lstm_preinject_vs_initinject(
    preinject_model=lstm_scratch,
    initinject_model=lstm_initinject,
    cnn_features=cnn_test[:5],
    idx2word=idx2word,
    max_length=SEQ_LENGTH,
    n_samples=5,
)

# ---------- BLEU-4 evaluation ----------
print("\n[Bonus] Evaluasi BLEU untuk LSTM Init-Inject pada test set...")
_ii_metrics, _ii_preds = evaluate_bleu_lstm_initinject(
    lstm_keras_initinject, cnn_test, gt_captions_test, idx2word,
    max_length=SEQ_LENGTH, verbose=True,
)

# Load Pre-Inject BLEU
_lstm_m_path = _os.path.join(_os.path.dirname(_os.path.abspath(".")), "results", "rnn_lstm", "lstm_metrics.json")
if "lstm_metrics" in globals():
    _pre_metrics = lstm_metrics
elif _os.path.exists(_lstm_m_path):
    with open(_lstm_m_path) as _f:
        _pre_metrics = _json.load(_f)
    print(f"[Bonus] Pre-Inject BLEU dimuat dari cache.")
else:
    # Evaluasi ulang dari Keras pre-inject model (BEST_LSTM_MODEL)
    if "BEST_LSTM_MODEL" in globals():
        from lstm.keras.evaluate import evaluate_model as _lstm_eval
        print("[Bonus] Evaluasi ulang LSTM Pre-Inject...")
        _pre_metrics, _ = _lstm_eval(
            model=BEST_LSTM_MODEL,
            cnn_features=cnn_test,
            gt_captions=gt_captions_test,
            idx2word=idx2word,
            max_length=SEQ_LENGTH,
            batch_size=64,
            verbose=True,
        )
        _os.makedirs(_os.path.dirname(_lstm_m_path), exist_ok=True)
        with open(_lstm_m_path, "w") as _f:
            _json.dump(_pre_metrics, _f, indent=2)
    else:
        print("[WARN] BEST_LSTM_MODEL tidak tersedia — Pre-Inject BLEU tidak bisa dihitung.")
        _pre_metrics = {}

# ---------- Comparison table ----------
print()
print("=" * 62)
print(f"{'LSTM Pre-Inject vs Init-Inject — BLEU Comparison':^62}")
print("=" * 62)
print(f"{'Metric':<15} {'Pre-Inject':>15} {'Init-Inject':>15} {'Diff':>15}")
print("-" * 62)
for _key in ["bleu1", "bleu2", "bleu3", "bleu4", "meteor"]:
    _pre_val = _pre_metrics.get(_key, 0.0)
    _ii_val  = _ii_metrics.get(_key, 0.0)
    _diff    = _ii_val - _pre_val
    print(f"{_key.upper():<15} {_pre_val:>15.4f} {_ii_val:>15.4f} {_diff:>+15.4f}")
print("=" * 62)
lstm_initinject_metrics = _ii_metrics


LSTMInitInject dibangun: hidden_dim=512, num_layers=1
[InitInject] Bobot embedding + LSTM + output_dense (LSTM-half) berhasil ditransfer.
[Bonus] Belum ada bobot Keras Init-Inject — melatih sekarang...

[Training] lstm_l1_h512_initinject
  embed_dim=256, hidden_dim=512, num_layers=1, epochs=30, batch_size=64
Epoch 1/30
132/506 [======>.......................] - ETA: 6:10 - loss: 2.0684 - accuracy: 0.6841

## Bagian 6 — Evaluasi & Perbandingan RNN vs LSTM

Evaluasi dilakukan dalam beberapa langkah:
1. **Ground truth** — dekode `lbl_test` ke caption string
2. **Keras RNN & LSTM** — evaluasi model Keras terbaik di test set (BLEU-1/2/3/4)
3. **Scratch RNN & LSTM** — muat bobot Keras ke implementasi NumPy, jalankan greedy decode
4. **Tabel perbandingan** — BLEU-1/2/3/4 untuk semua model
5. **Contoh kualitatif** — caption Ground Truth vs Keras-RNN vs Keras-LSTM vs Scratch-RNN vs Scratch-LSTM
6. **Training curves** — plot loss per epoch untuk model terbaik

In [4]:
# -- Guard: reconstruct lbl_test / idx2word when running B6 standalone --------
import os as _os, json as _json, sys as _sys
import numpy as _np

if 'idx2word' not in globals() or 'lbl_test' not in globals():
    _proj_root = _os.path.dirname(_os.path.abspath('.'))

    # Ensure shared modules are importable
    if not any(_os.path.isdir(_os.path.join(p, 'shared')) for p in _sys.path):
        for _cand in [
            '/content/ChosaHeidan_Tubes-2_IF3270/src',
            _os.path.join(_os.path.abspath('.'), 'src'),
            _os.path.abspath('.'),
        ]:
            if _os.path.isdir(_os.path.join(_cand, 'shared')):
                _sys.path.insert(0, _cand)
                break

    if 'idx2word' not in globals():
        _idx2word_path = _os.path.join(_proj_root, 'data', 'idx2word.json')
        with open(_idx2word_path, encoding='utf-8') as _f:
            idx2word = {int(k): v for k, v in _json.load(_f).items()}
        print(f'[B6-guard] idx2word loaded --- {len(idx2word)} words')

    if 'lbl_test' not in globals():
        from shared.caption_preprocess import split_captions_file, prepare_training_data

        if 'word2idx' not in globals():
            _w2i_path = _os.path.join(_proj_root, 'data', 'word2idx.json')
            with open(_w2i_path, encoding='utf-8') as _f:
                word2idx = _json.load(_f)
        if 'test_ids' not in globals():
            _test_ids_path = _os.path.join(_proj_root, 'data', 'test_ids.txt')
            with open(_test_ids_path, encoding='utf-8') as _f:
                test_ids = [l.strip() for l in _f if l.strip()]
        if 'captions_dict' not in globals():
            _captions_path = _os.path.join(_proj_root, 'data', 'flickr8k', 'captions.txt')
            captions_dict = split_captions_file(_captions_path)
        if 'features_dict' not in globals():
            from shared.feature_extract import load_cnn_features
            _feat_path     = _os.path.join(_proj_root, 'data', 'flickr8k_inception.npy')
            _feat_ids_path = _os.path.join(_proj_root, 'data', 'flickr8k_image_ids.json')
            _raw_feats = load_cnn_features(_feat_path)
            with open(_feat_ids_path, encoding='utf-8') as _f:
                _feat_ids = _json.load(_f)
            features_dict = {img_id: _raw_feats[i] for i, img_id in enumerate(_feat_ids)}
        if 'MAX_LENGTH' not in globals():
            MAX_LENGTH = 35
        # Rebuild test split arrays (mirrors nb02-b3 logic)
        _matched_ids, _full_seqs = prepare_training_data(
            captions_dict, test_ids, word2idx,
            max_length=MAX_LENGTH, add_start=True, add_end=True)
        _valid = [(i, img_id) for i, img_id in enumerate(_matched_ids) if img_id in features_dict]
        _idxs        = [v[0] for v in _valid]
        test_img_ids = [v[1] for v in _valid]
        _seqs    = _full_seqs[_idxs]
        cnn_test = _np.array([features_dict[i] for i in test_img_ids])
        seq_test = _seqs[:, :-1]
        lbl_test = _seqs[:, 1:]
        print(f'[B6-guard] lbl_test rebuilt --- {lbl_test.shape[0]} rows')

# 6a. decode target sequences => ground truth caption strings
def decode_target_seqs(target_seqs, idx2word):
    """Decode lbl_test rows -> list of caption string."""
    END_WORD = '<end>'
    SKIP = {'<start>', '<end>', '<pad>', ''}
    captions = []
    for seq in target_seqs:
        words = []
        for t in seq:
            w = idx2word.get(int(t), '')
            if w == END_WORD:
                break
            if w not in SKIP:
                words.append(w)
        captions.append(' '.join(words))
    return captions

gt_captions_test = decode_target_seqs(lbl_test, idx2word)
print(f'[B6a] Ground truth captions (test) : {len(gt_captions_test)}')
print(f'      Contoh [0] : {gt_captions_test[0]}')


[B6-guard] idx2word loaded --- 2653 words
[Feature Extraction] Features dimuat dari: c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\data\flickr8k_inception.npy (shape: (8091, 2048))
[B6-guard] lbl_test rebuilt --- 4050 rows
[B6a] Ground truth captions (test) : 4050
      Contoh [0] : a girl with a white backpack is standing and smaller children are sitting in a row on the ground


In [ ]:
import os as _os, json as _json, sys as _sys
import numpy as _np

if any(v not in globals() for v in ['lbl_test', 'idx2word', 'cnn_test', 'gt_captions_test']):
    _proj_root = _os.path.dirname(_os.path.abspath('.'))
    if not any(_os.path.isdir(_os.path.join(p, 'shared')) for p in _sys.path):
        for _cand in [
            '/content/ChosaHeidan_Tubes-2_IF3270/src',
            _os.path.join(_os.path.abspath('.'), 'src'),
            _os.path.abspath('.'),
        ]:
            if _os.path.isdir(_os.path.join(_cand, 'shared')):
                _sys.path.insert(0, _cand)
                break
    if 'idx2word' not in globals():
        _idx2word_path = _os.path.join(_proj_root, 'data', 'idx2word.json')
        with open(_idx2word_path, encoding='utf-8') as _f:
            idx2word = {int(k): v for k, v in _json.load(_f).items()}
        print(f'[B6-guard] idx2word loaded --- {len(idx2word)} words')
    if 'lbl_test' not in globals() or 'cnn_test' not in globals():
        from shared.caption_preprocess import split_captions_file, prepare_training_data
        if 'word2idx' not in globals():
            with open(_os.path.join(_proj_root, 'data', 'word2idx.json'), encoding='utf-8') as _f:
                word2idx = _json.load(_f)
        if 'test_ids' not in globals():
            with open(_os.path.join(_proj_root, 'data', 'test_ids.txt'), encoding='utf-8') as _f:
                test_ids = [l.strip() for l in _f if l.strip()]
        if 'captions_dict' not in globals():
            captions_dict = split_captions_file(_os.path.join(_proj_root, 'data', 'flickr8k', 'captions.txt'))
        if 'features_dict' not in globals():
            from shared.feature_extract import load_cnn_features
            _raw_feats = load_cnn_features(_os.path.join(_proj_root, 'data', 'flickr8k_inception.npy'))
            with open(_os.path.join(_proj_root, 'data', 'flickr8k_image_ids.json'), encoding='utf-8') as _f:
                _feat_ids = _json.load(_f)
            features_dict = {img_id: _raw_feats[i] for i, img_id in enumerate(_feat_ids)}
        if 'MAX_LENGTH' not in globals():
            MAX_LENGTH = 35
        _matched_ids, _full_seqs = prepare_training_data(
            captions_dict, test_ids, word2idx,
            max_length=MAX_LENGTH, add_start=True, add_end=True)
        _valid = [(i, img_id) for i, img_id in enumerate(_matched_ids) if img_id in features_dict]
        _idxs        = [v[0] for v in _valid]
        test_img_ids = [v[1] for v in _valid]
        _seqs    = _full_seqs[_idxs]
        cnn_test = _np.array([features_dict[i] for i in test_img_ids])
        seq_test = _seqs[:, :-1]
        lbl_test = _seqs[:, 1:]
        print(f'[B6-guard] lbl_test rebuilt --- {lbl_test.shape[0]} rows')
    if 'gt_captions_test' not in globals():
        _END_WORD = '<end>'
        _SKIP = {'<start>', '<end>', '<pad>', ''}
        gt_captions_test = []
        for _seq in lbl_test:
            _words = []
            for _t in _seq:
                _w = idx2word.get(int(_t), '')
                if _w == _END_WORD:
                    break
                if _w not in _SKIP:
                    _words.append(_w)
            gt_captions_test.append(' '.join(_words))
        print(f'[B6-guard] gt_captions_test rebuilt --- {len(gt_captions_test)} captions')

import os as _os, json as _json, re as _re
import tensorflow as _tf

if 'SEQ_LENGTH' not in globals():
    SEQ_LENGTH = 34
if 'VOCAB_SIZE' not in globals():
    VOCAB_SIZE = 2653
if 'FEATURE_DIM' not in globals():
    FEATURE_DIM = 2048

if 'BEST_RNN_MODEL' not in globals():
    _proj_root = _os.path.dirname(_os.path.abspath('.'))
    _src = _os.path.abspath('.')
    if not _os.path.exists(_os.path.join(_src, 'rnn')):
        _src = _os.path.join(_src, 'src')
    import sys as _sys
    if _src not in _sys.path:
        _sys.path.insert(0, _src)

    from rnn.keras.model_keras import build_rnn_decoder_preinject

    if 'RNN_WEIGHTS_DIR' not in globals():
        RNN_WEIGHTS_DIR = _os.path.join(_proj_root, 'weights', 'rnn')
    if 'RESULTS_DIR' not in globals():
        RESULTS_DIR = _os.path.join(_proj_root, 'results', 'rnn_lstm')
        _os.makedirs(RESULTS_DIR, exist_ok=True)

    if 'rnn_comparison' not in globals():
        _rnn_cmp_json = _os.path.join(RESULTS_DIR, 'rnn_comparison.json')
        if _os.path.exists(_rnn_cmp_json):
            with open(_rnn_cmp_json) as _f:
                rnn_comparison = _json.load(_f)
            print(f'[B6b-guard] rnn_comparison loaded --- {len(rnn_comparison)} models')
        else:
            # Build from *_history.json files in weights/rnn/
            _hist_files = sorted([
                fn for fn in _os.listdir(RNN_WEIGHTS_DIR)
                if fn.endswith('_history.json')
            ]) if _os.path.isdir(RNN_WEIGHTS_DIR) else []
            if not _hist_files:
                raise RuntimeError(
                    'rnn_comparison.json tidak ada dan tidak ada *_history.json '
                    f'di {RNN_WEIGHTS_DIR} -- jalankan cell nb02-b4 terlebih dahulu.'
                )
            rnn_comparison = {}
            for _hf in _hist_files:
                _name = _hf.replace('_history.json', '')
                with open(_os.path.join(RNN_WEIGHTS_DIR, _hf)) as _f:
                    _hist = _json.load(_f)
                _vl = _hist.get('val_loss', [])
                rnn_comparison[_name] = {
                    'best_val_loss': float(min(_vl)) if _vl else None,
                    'epochs_trained': len(_hist.get('loss', [])),
                }
            with open(_rnn_cmp_json, 'w') as _f:
                _json.dump(rnn_comparison, _f, indent=2)
            print(f'[B6b-guard] rnn_comparison built from {len(_hist_files)} history files')

    _rnn_ranked = sorted(
        [(k, v) for k, v in rnn_comparison.items() if v.get('best_val_loss') is not None],
        key=lambda x: x[1]['best_val_loss'],
    )
    BEST_RNN_NAME = _rnn_ranked[0][0]

    def _load_keras3_weights(model, path):
        import h5py as _h5
        with _h5.File(path, 'r') as _f:
            if 'layers' not in _f:
                raise ValueError(f'Unknown .weights.h5 format in {path}')
            _lg = _f['layers']
            _cnt = {}
            for _layer in model.layers:
                _cls = type(_layer).__name__.lower()
                _n = _cnt.get(_cls, 0)
                _cnt[_cls] = _n + 1
                _key = _cls if _n == 0 else f'{_cls}_{_n}'
                if _key not in _lg:
                    continue
                _grp = _lg[_key]
                if 'vars' in _grp and len(_grp['vars']) > 0:
                    for _i, _v in enumerate(_layer.variables):
                        if str(_i) in _grp['vars']:
                            _v.assign(_grp['vars'][str(_i)][()])
                if 'cell' in _grp and hasattr(_layer, 'cell'):
                    _cgrp = _grp['cell']
                    if 'vars' in _cgrp:
                        for _i, _v in enumerate(_layer.cell.variables):
                            if str(_i) in _cgrp['vars']:
                                _v.assign(_cgrp['vars'][str(_i)][()])
                if 'layer' in _grp and hasattr(_layer, 'layer'):
                    _igrp = _grp['layer']
                    if 'vars' in _igrp:
                        for _i, _v in enumerate(_layer.layer.variables):
                            if str(_i) in _igrp['vars']:
                                _v.assign(_igrp['vars'][str(_i)][()])

    _m = _re.match(r'rnn_l(\d+)_h(\d+)_preinject', BEST_RNN_NAME)
    _num_layers = int(_m.group(1))
    _hidden_dim = int(_m.group(2))

    BEST_RNN_MODEL = build_rnn_decoder_preinject(
        vocab_size=VOCAB_SIZE, embed_dim=256, hidden_dim=_hidden_dim,
        num_layers=_num_layers, feature_dim=FEATURE_DIM,
        seq_max_length=SEQ_LENGTH, dropout=0.3,
    )
    BEST_RNN_MODEL([
        _tf.zeros((1, FEATURE_DIM)),
        _tf.zeros((1, SEQ_LENGTH), dtype=_tf.int32),
    ])

    _best_h5  = _os.path.join(RNN_WEIGHTS_DIR, f'{BEST_RNN_NAME}_best.h5')
    _best_wh5 = _os.path.join(RNN_WEIGHTS_DIR, f'{BEST_RNN_NAME}_best.weights.h5')
    if _os.path.exists(_best_h5):
        BEST_RNN_MODEL.load_weights(_best_h5)
        print(f'[B6b-guard] Bobot dimuat dari: {_best_h5}')
    elif _os.path.exists(_best_wh5):
        _load_keras3_weights(BEST_RNN_MODEL, _best_wh5)
        print(f'[B6b-guard] Bobot dimuat dari: {_best_wh5}')
    else:
        raise FileNotFoundError(f'Tidak ditemukan: {_best_h5} maupun {_best_wh5}')
    print(f'[B6b-guard] BEST_RNN_MODEL ready: {BEST_RNN_NAME}')

from rnn.keras.evaluate import evaluate_model as rnn_evaluate_keras

_rnn_metrics_path = _os.path.join(
    _os.path.dirname(_os.path.abspath('.')), 'results', 'rnn_lstm', 'rnn_metrics.json')
_rnn_preds_path   = _os.path.join(
    _os.path.dirname(_os.path.abspath('.')), 'results', 'rnn_lstm', 'rnn_preds.json')

if 'rnn_metrics' not in globals() or 'rnn_preds' not in globals():
    if _os.path.exists(_rnn_metrics_path) and _os.path.exists(_rnn_preds_path):
        with open(_rnn_metrics_path) as _f:
            rnn_metrics = _json.load(_f)
        with open(_rnn_preds_path) as _f:
            rnn_preds = _json.load(_f)
        print(f'[B6b] rnn_metrics dimuat dari cache: {_rnn_metrics_path}')
    else:
        print('[B6b] Evaluasi Keras RNN terbaik pada test set...')
        rnn_metrics, rnn_preds = rnn_evaluate_keras(
            model=BEST_RNN_MODEL,
            cnn_features=cnn_test,
            gt_captions=gt_captions_test,
            idx2word=idx2word,
            max_length=SEQ_LENGTH,
            batch_size=64,
            verbose=True,
        )
        _os.makedirs(_os.path.dirname(_rnn_metrics_path), exist_ok=True)
        with open(_rnn_metrics_path, 'w') as _f:
            _json.dump(rnn_metrics, _f, indent=2)
        with open(_rnn_preds_path, 'w') as _f:
            _json.dump(rnn_preds, _f, indent=2)
        print(f'[B6b] Hasil disimpan ke: {_rnn_metrics_path}')
else:
    print('[B6b] rnn_metrics sudah ada di globals -- skip.')
print(f'  RNN  BLEU-1: {rnn_metrics.get("bleu1",0):.4f}')
print(f'  RNN  BLEU-2: {rnn_metrics.get("bleu2",0):.4f}')
print(f'  RNN  BLEU-3: {rnn_metrics.get("bleu3",0):.4f}')
print(f'  RNN  BLEU-4: {rnn_metrics.get("bleu4",0):.4f}')


[B6b] Evaluasi Keras RNN terbaik pada test set...
  Step 5/33  (0/4050 selesai)
  Step 10/33  (1615/4050 selesai)
  Step 15/33  (4050/4050 selesai)
[B6b] Hasil disimpan ke: c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\results\rnn_lstm\rnn_metrics.json
  RNN  BLEU-1: 0.2679
  RNN  BLEU-2: 0.1337
  RNN  BLEU-3: 0.0703
  RNN  BLEU-4: 0.0402


In [2]:
import importlib
import lstm.keras.evaluate as _lstm_eval
importlib.reload(_lstm_eval)
from lstm.keras.evaluate import evaluate_model as lstm_evaluate_keras


In [9]:
from lstm.keras.evaluate import evaluate_model as lstm_evaluate_keras

print('\n[B6c] Evaluasi Keras LSTM terbaik pada test set...')
lstm_metrics, lstm_preds = lstm_evaluate_keras(
    model=BEST_LSTM_MODEL,
    cnn_features=cnn_test,
    gt_captions=gt_captions_test,
    idx2word=idx2word,
    max_length=SEQ_LENGTH,
    batch_size=64,
    verbose=True,
)
print(f'\n  LSTM BLEU-1: {lstm_metrics.get("bleu1",0):.4f}')
print(f'  LSTM BLEU-2: {lstm_metrics.get("bleu2",0):.4f}')
print(f'  LSTM BLEU-3: {lstm_metrics.get("bleu3",0):.4f}')
print(f'  LSTM BLEU-4: {lstm_metrics.get("bleu4",0):.4f}')


import os as _os, json as _json
_proj_root = _os.path.dirname(_os.path.abspath('.'))
_lstm_preds_path   = _os.path.join(_proj_root, 'results', 'rnn_lstm', 'lstm_preds.json')
_lstm_metrics_path = _os.path.join(_proj_root, 'results', 'rnn_lstm', 'lstm_metrics.json')
_os.makedirs(_os.path.dirname(_lstm_preds_path), exist_ok=True)
with open(_lstm_preds_path, 'w', encoding='utf-8') as _f:
    _json.dump(lstm_preds, _f, ensure_ascii=False, indent=2)
with open(_lstm_metrics_path, 'w', encoding='utf-8') as _f:
    _json.dump(lstm_metrics, _f, ensure_ascii=False, indent=2)
print(f'[B6c] lstm_preds cached: {_lstm_preds_path}')


[B6c] Evaluasi Keras LSTM terbaik pada test set...


NameError: name 'BEST_LSTM_MODEL' is not defined

In [25]:
import re as _re
from rnn.scratch.model_scratch import RNNScratch

print('[B6d] Scratch RNN — muat bobot Keras .h5 ke implementasi NumPy...')

if rnn_results is not None:
    best_rnn_cfg = rnn_results[BEST_RNN_NAME]['config']
else:
    _m = _re.match(r'rnn_l(\d+)_h(\d+)_preinject', BEST_RNN_NAME)
    best_rnn_cfg = {
        'num_layers': int(_m.group(1)),
        'hidden_dim': int(_m.group(2)),
        'embed_dim': 256,
        'feature_dim': FEATURE_DIM,
    }

rnn_scratch = RNNScratch(
    vocab_size=VOCAB_SIZE,
    embed_dim=best_rnn_cfg['embed_dim'],
    hidden_dim=best_rnn_cfg['hidden_dim'],
    num_layers=best_rnn_cfg['num_layers'],
    feature_dim=FEATURE_DIM,
)
rnn_scratch.build()

best_rnn_h5 = os.path.join(RNN_WEIGHTS_DIR, f'{BEST_RNN_NAME}_best.weights.h5')
if os.path.exists(best_rnn_h5):
    rnn_scratch.load_weights_from_h5(best_rnn_h5)
    print(f'  Bobot dimuat dari: {best_rnn_h5}')
else:
    print(f'  [WARN] File tidak ditemukan: {best_rnn_h5}')
    print('         Jalankan Bagian 4 terlebih dahulu.')

# Greedy decode 5 sample dari test set
N_SAMPLE = 5
sample_feats = cnn_test[:N_SAMPLE]
scratch_rnn_caps = rnn_scratch.greedy_decode_batch(
    sample_feats, idx2word, max_length=SEQ_LENGTH
)
print(f'[B6d] Scratch RNN greedy decode ({N_SAMPLE} sample):')
for i, cap in enumerate(scratch_rnn_caps):
    print(f'  [{i+1}] {cap}')

[B6d] Scratch RNN — muat bobot Keras .h5 ke implementasi NumPy...
Bobot berhasil dimuat dari: c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\weights\rnn\rnn_l1_h512_preinject_best.weights.h5
  Bobot dimuat dari: c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\weights\rnn\rnn_l1_h512_preinject_best.weights.h5
[B6d] Scratch RNN greedy decode (5 sample):
  [1] colored sale inflatable garden groom snowboard carts begging plaza observing racers leans wheels artificial parked countryside things good similar surfer area dot shovels bathtub leafy costumes pack construction single baseball check dolphin ledge muscular
  [2] colored sale inflatable garden groom snowboard carts begging plaza observing racers leans wheels artificial parked countryside things good similar surfer area dot shovels bathtub leafy costumes pack construct

In [27]:
from lstm.scratch.model_scratch import LSTMScratch
import json as _json, os as _os, re as _re

print("[B6e] Scratch LSTM - muat bobot Keras .h5 ke implementasi NumPy...")

if 'FEATURE_DIM' not in globals():
    FEATURE_DIM = 2048
if 'VOCAB_SIZE' not in globals():
    VOCAB_SIZE = 2653
if 'SEQ_LENGTH' not in globals():
    SEQ_LENGTH = 34

if 'lstm_results' not in globals():
    lstm_results = None

if 'BEST_LSTM_NAME' not in globals():
    _proj_root = _os.path.dirname(_os.path.abspath('.'))
    _cmp_json = _os.path.join(_proj_root, 'results', 'rnn_lstm', 'lstm_comparison.json')
    with open(_cmp_json) as _f:
        _cmp = _json.load(_f)
    _ranked = sorted(
        [(k, v) for k, v in _cmp.items() if v.get('best_val_loss') is not None],
        key=lambda x: x[1]['best_val_loss'],
    )
    BEST_LSTM_NAME = _ranked[0][0]
    print(f'[B6e] BEST_LSTM_NAME rekonstruksi: {BEST_LSTM_NAME}')

if lstm_results is not None:
    best_lstm_cfg = lstm_results[BEST_LSTM_NAME]['config']
else:
    _m = _re.match(r'lstm_l(\d+)_h(\d+)_preinject', BEST_LSTM_NAME)
    best_lstm_cfg = {
        'num_layers': int(_m.group(1)),
        'hidden_dim': int(_m.group(2)),
        'embed_dim': 256,
        'feature_dim': FEATURE_DIM,
    }

lstm_scratch = LSTMScratch(
    vocab_size=VOCAB_SIZE,
    embed_dim=best_lstm_cfg['embed_dim'],
    hidden_dim=best_lstm_cfg['hidden_dim'],
    num_layers=best_lstm_cfg['num_layers'],
    feature_dim=FEATURE_DIM,
)
lstm_scratch.build()

if 'LSTM_WEIGHTS_DIR' not in globals():
    LSTM_WEIGHTS_DIR = _os.path.join(_os.path.abspath('.'), 'weights', 'lstm')

best_lstm_h5  = _os.path.join(LSTM_WEIGHTS_DIR, f'{BEST_LSTM_NAME}_best.h5')
best_lstm_wh5 = _os.path.join(LSTM_WEIGHTS_DIR, f'{BEST_LSTM_NAME}_best.weights.h5')
if _os.path.exists(best_lstm_h5):
    lstm_scratch.load_weights_from_h5(best_lstm_h5)
    print(f'  Bobot dimuat dari: {best_lstm_h5}')
elif _os.path.exists(best_lstm_wh5):
    lstm_scratch.load_weights_from_h5(best_lstm_wh5)
    print(f'  Bobot dimuat dari: {best_lstm_wh5}')
else:
    print(f'  [WARN] File tidak ditemukan: {best_lstm_h5}')
    print('         Jalankan Bagian 5 terlebih dahulu.')

if 'sample_feats' not in globals() or 'idx2word' not in globals() or 'N_SAMPLE' not in globals():
    print('[WARN] sample_feats / idx2word / N_SAMPLE belum terdefinisi — skip decode.')
else:
    scratch_lstm_caps = lstm_scratch.greedy_decode_batch(
        sample_feats, idx2word, max_length=SEQ_LENGTH
    )
    print(f'[B6e] Scratch LSTM greedy decode ({N_SAMPLE} sample):')
    for i, cap in enumerate(scratch_lstm_caps):
        print(f'  [{i+1}] {cap}')

[B6e] Scratch LSTM - muat bobot Keras .h5 ke implementasi NumPy...
Bobot berhasil dimuat dari: c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\weights\lstm\lstm_l1_h512_preinject_best.weights.h5
  Bobot dimuat dari: c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\weights\lstm\lstm_l1_h512_preinject_best.weights.h5
[B6e] Scratch LSTM greedy decode (5 sample):
  [1] camel suspended sing musician boston sharp saying ducks walkway lab strap trophy skyline abandoned hamburgers directions patches monkey including flannel headdress lifts time robes floral cheek cheek sporting jumper ninja legged farm bank seagull
  [2] camel suspended sing musician boston sharp saying ducks walkway lab strap trophy skyline abandoned hamburgers directions patches monkey including flannel headdress lifts time robes floral cheek cheek sporting jum

In [28]:
import json as _json

METRICS = ['bleu1', 'bleu2', 'bleu3', 'bleu4']

print(f'\n{"="*70}')
print('  PERBANDINGAN BLEU — Keras RNN vs Keras LSTM — Test Set')
print(f'{"="*70}')
print(f'  {"Metric":<15} {"RNN":>12} {"LSTM":>12} {"Winner":>10}')
print(f'  {"-"*52}')

for m in METRICS:
    rv = rnn_metrics.get(m, 0.0)
    lv = lstm_metrics.get(m, 0.0)
    winner = 'LSTM' if lv > rv else ('RNN' if rv > lv else 'tie')
    print(f'  {m.upper():<15} {rv:>12.4f} {lv:>12.4f} {winner:>10}')

avg_rnn  = sum(rnn_metrics.get(m, 0)  for m in METRICS) / 4
avg_lstm = sum(lstm_metrics.get(m, 0) for m in METRICS) / 4
winner   = 'LSTM' if avg_lstm > avg_rnn else 'RNN'
print(f'  {"-"*52}')
print(f'  {"Avg BLEU":<15} {avg_rnn:>12.4f} {avg_lstm:>12.4f} {winner:>10}')
print(f'{"="*70}')

# Simpan metrik
metrics_out = {
    'rnn':  {'model': BEST_RNN_NAME,  **rnn_metrics},
    'lstm': {'model': BEST_LSTM_NAME, **lstm_metrics},
}
eval_path = os.path.join(RESULTS_DIR, 'rnn_lstm_eval.json')
with open(eval_path, 'w') as f:
    _json.dump(metrics_out, f, indent=2)
print(f'\n[B6f] Metrik disimpan ke: {eval_path}')



  PERBANDINGAN BLEU — Keras RNN vs Keras LSTM — Test Set
  Metric                   RNN         LSTM     Winner
  ----------------------------------------------------
  BLEU1                 0.2679       0.2992       LSTM
  BLEU2                 0.1337       0.1646       LSTM
  BLEU3                 0.0703       0.0942       LSTM
  BLEU4                 0.0402       0.0555       LSTM
  ----------------------------------------------------
  Avg BLEU              0.1280       0.1534       LSTM

[B6f] Metrik disimpan ke: c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\results\rnn_lstm\rnn_lstm_eval.json


In [8]:
# ── Contoh kualitatif: GT vs Keras-RNN vs Keras-LSTM vs Scratch-RNN vs Scratch-LSTM
from shared.plot_utils import plot_caption_samples
import os as _os, json as _json, sys as _sys
import numpy as _np

_proj_root = _os.path.dirname(_os.path.abspath('.'))

# Ensure shared/ is importable
if not any(_os.path.isdir(_os.path.join(p, 'shared')) for p in _sys.path):
    for _cand in [_os.path.abspath('.'), _os.path.join(_os.path.abspath('.'), 'src')]:
        if _os.path.isdir(_os.path.join(_cand, 'shared')):
            _sys.path.insert(0, _cand)
            break

# Reconstruct path constants
if 'FLICKR8K_DIR' not in globals():
    FLICKR8K_DIR = _os.path.join(_proj_root, 'data', 'flickr8k')
if 'IMAGE_DIR' not in globals():
    IMAGE_DIR = _os.path.join(FLICKR8K_DIR, 'Images')
if 'RESULTS_DIR' not in globals():
    RESULTS_DIR = _os.path.join(_proj_root, 'results', 'rnn_lstm')
    _os.makedirs(RESULTS_DIR, exist_ok=True)

# Reconstruct test_img_ids / gt_captions_test from data files if needed
if any(v not in globals() for v in ['test_img_ids', 'gt_captions_test', 'lbl_test']):
    from shared.caption_preprocess import split_captions_file, prepare_training_data
    if 'word2idx' not in globals():
        with open(_os.path.join(_proj_root, 'data', 'word2idx.json'), encoding='utf-8') as _f:
            word2idx = _json.load(_f)
    if 'idx2word' not in globals():
        with open(_os.path.join(_proj_root, 'data', 'idx2word.json'), encoding='utf-8') as _f:
            idx2word = {int(k): v for k, v in _json.load(_f).items()}
    if 'test_ids' not in globals():
        with open(_os.path.join(_proj_root, 'data', 'test_ids.txt'), encoding='utf-8') as _f:
            test_ids = [l.strip() for l in _f if l.strip()]
    if 'captions_dict' not in globals():
        captions_dict = split_captions_file(_os.path.join(FLICKR8K_DIR, 'captions.txt'))
    if 'features_dict' not in globals():
        from shared.feature_extract import load_cnn_features
        _raw = load_cnn_features(_os.path.join(_proj_root, 'data', 'flickr8k_inception.npy'))
        with open(_os.path.join(_proj_root, 'data', 'flickr8k_image_ids.json'), encoding='utf-8') as _f:
            _ids = _json.load(_f)
        features_dict = {img_id: _raw[i] for i, img_id in enumerate(_ids)}
    if 'MAX_LENGTH' not in globals():
        MAX_LENGTH = 35
    _matched_ids, _full_seqs = prepare_training_data(
        captions_dict, test_ids, word2idx,
        max_length=MAX_LENGTH, add_start=True, add_end=True)
    _valid = [(i, img_id) for i, img_id in enumerate(_matched_ids) if img_id in features_dict]
    _idxs        = [v[0] for v in _valid]
    test_img_ids = [v[1] for v in _valid]
    _seqs = _full_seqs[_idxs]
    lbl_test = _seqs[:, 1:]
    print(f'[qual-guard] test_img_ids rebuilt --- {len(test_img_ids)} images')
    if 'gt_captions_test' not in globals():
        _SKIP = {'<start>', '<end>', '<pad>', ''}
        gt_captions_test = []
        for _seq in lbl_test:
            _words = []
            for _t in _seq:
                _w = idx2word.get(int(_t), '')
                if _w == '<end>':
                    break
                if _w not in _SKIP:
                    _words.append(_w)
            gt_captions_test.append(' '.join(_words))
        print(f'[qual-guard] gt_captions_test rebuilt --- {len(gt_captions_test)} captions')

# Load rnn_preds from JSON cache if needed
if 'rnn_preds' not in globals():
    _p = _os.path.join(_proj_root, 'results', 'rnn_lstm', 'rnn_preds.json')
    if _os.path.exists(_p):
        with open(_p, encoding='utf-8') as _f:
            rnn_preds = _json.load(_f)
        print(f'[qual-guard] rnn_preds loaded from cache ({len(rnn_preds)} preds)')
    else:
        raise RuntimeError(f'rnn_preds.json tidak ditemukan: {_p}. Jalankan sel B6b terlebih dahulu.')

# Load lstm_preds from JSON cache if needed
if 'lstm_preds' not in globals():
    _p = _os.path.join(_proj_root, 'results', 'rnn_lstm', 'lstm_preds.json')
    if _os.path.exists(_p):
        with open(_p, encoding='utf-8') as _f:
            lstm_preds = _json.load(_f)
        print(f'[qual-guard] lstm_preds loaded from cache ({len(lstm_preds)} preds)')
    else:
        print(f'[qual-guard] WARN: lstm_preds.json tidak ditemukan -- LSTM akan dilewati.')
        lstm_preds = []

# Fallback for scratch caps if cells B6d/B6e were skipped
if 'scratch_rnn_caps' not in globals():
    scratch_rnn_caps = []
if 'scratch_lstm_caps' not in globals():
    scratch_lstm_caps = []

N_SHOW = 5
sample_img_ids   = test_img_ids[:N_SHOW]
sample_img_paths = [_os.path.join(IMAGE_DIR, img_id) for img_id in sample_img_ids]

print(f'{"="*70}')
print(f'  CONTOH CAPTION ({N_SHOW} sample dari test set)')
print(f'{"="*70}')

for idx in range(N_SHOW):
    print(f'  [{idx+1}] Gambar: {sample_img_ids[idx]}')
    print(f'       GT           : {gt_captions_test[idx]}')
    print(f'       Keras RNN    : {rnn_preds[idx]}')
    print(f'       Keras LSTM   : {lstm_preds[idx]}')
    print(f'       Scratch RNN  : {scratch_rnn_caps[idx] if idx < len(scratch_rnn_caps) else "-"}')
    print(f'       Scratch LSTM : {scratch_lstm_caps[idx] if idx < len(scratch_lstm_caps) else "-"}')

# Plot qualitative dengan gambar asli (GT vs Keras RNN)
plot_caption_samples(
    image_paths=sample_img_paths,
    gt_captions=gt_captions_test[:N_SHOW],
    pred_captions=rnn_preds[:N_SHOW],
    save_path=_os.path.join(RESULTS_DIR, 'qualitative_rnn.png'),
    n=N_SHOW,
)
# Plot LSTM captions
if lstm_preds:
    plot_caption_samples(
        image_paths=sample_img_paths,
        gt_captions=gt_captions_test[:N_SHOW],
        pred_captions=lstm_preds[:N_SHOW],
        save_path=_os.path.join(RESULTS_DIR, 'qualitative_lstm.png'),
        n=N_SHOW,
    )
else:
    print('[qual] LSTM plot dilewati (lstm_preds kosong).')

[qual-guard] test_img_ids rebuilt --- 4050 images
[qual-guard] rnn_preds loaded from cache (4050 preds)


RuntimeError: lstm_preds.json tidak ditemukan: c:\Users\HYPE R Series\OneDrive - Institut Teknologi Bandung\Documents\ITB\Semester 6\Pembelajaran Mesin\ChosaHeidan_Tubes-2_IF3270\results\rnn_lstm\lstm_preds.json. Jalankan sel B6c terlebih dahulu.

In [ ]:
from shared.plot_utils import plot_training_history, plot_bleu_comparison

# ── Training curves untuk model terbaik RNN dan LSTM ──────────────────────────
for label, results, name in [
    ('RNN',  rnn_results,  BEST_RNN_NAME),
    ('LSTM', lstm_results, BEST_LSTM_NAME),
]:
    history = results[name]['history'].history
    # plot_training_history expects {'train_loss': [...], 'val_loss': [...]}
    hist_dict = {
        'train_loss': history.get('loss', []),
        'val_loss':   history.get('val_loss', []),
    }
    plot_training_history(
        history=hist_dict,
        title=f'{label} Training — {name}',
        save_path=os.path.join(RESULTS_DIR, f'training_curve_{label.lower()}.png'),
    )

# ── BLEU comparison bar chart ─────────────────────────────────────────────────
bleu_compare = {
    'RNN':  rnn_metrics,
    'LSTM': lstm_metrics,
}
plot_bleu_comparison(
    results_dict=bleu_compare,
    title='BLEU Score — RNN vs LSTM (Test Set)',
    save_path=os.path.join(RESULTS_DIR, 'bleu_comparison.png'),
)
print('[B6h] Training curves dan BLEU comparison plot selesai.')


## Bagian 6-Bonus-1 — Beam Search vs Greedy Decoding

**Beam search** (k=5) menjelajahi k kandidat caption secara bersamaan di setiap langkah,
sehingga menghasilkan caption yang secara global lebih baik dibanding greedy (k=1).

Perbandingan dilakukan pada **scratch RNN** dan **scratch LSTM** yang sudah dilatih.

In [ ]:
from rnn.bonus.bonus_beam_search import (
    beam_search, compare_beam_vs_greedy, beam_search_with_length_penalty
)
from lstm.bonus.bonus_beam_search import (
    beam_search_lstm, compare_lstm_beam_vs_greedy,
    beam_search_lstm_with_length_penalty
)

N_BEAM = 5   # jumlah sample untuk perbandingan

# ── RNN: Beam Search vs Greedy ────────────────────────────────────────────────
print('[Bonus-Beam] RNN: Beam Search (k=5) vs Greedy')
print('='*65)
rnn_beam_results = compare_beam_vs_greedy(
    model=rnn_scratch,
    cnn_features=cnn_test[:N_BEAM],
    idx2word=idx2word,
    k=5,
    max_length=SEQ_LENGTH,
    n_samples=N_BEAM,
)

# RNN dengan length penalty
print('\n[Bonus-Beam] RNN: Beam Search dengan Length Penalty (alpha=0.6)')
for i, feat in enumerate(cnn_test[:3]):
    cap_lp = beam_search_with_length_penalty(
        model=rnn_scratch,
        cnn_feature=feat,
        idx2word=idx2word,
        k=5,
        max_length=SEQ_LENGTH,
        alpha=0.6,
    )
    print(f'  [{i+1}] {cap_lp}')

# ── LSTM: Beam Search vs Greedy ───────────────────────────────────────────────
print('\n[Bonus-Beam] LSTM: Beam Search (k=5) vs Greedy')
print('='*65)
lstm_beam_results = compare_lstm_beam_vs_greedy(
    model=lstm_scratch,
    cnn_features=cnn_test[:N_BEAM],
    idx2word=idx2word,
    k=5,
    max_length=SEQ_LENGTH,
    n_samples=N_BEAM,
)

# LSTM dengan length penalty
print('\n[Bonus-Beam] LSTM: Beam Search dengan Length Penalty (alpha=0.6)')
for i, feat in enumerate(cnn_test[:3]):
    cap_lp = beam_search_lstm_with_length_penalty(
        model=lstm_scratch,
        cnn_feature=feat,
        idx2word=idx2word,
        k=5,
        max_length=SEQ_LENGTH,
        alpha=0.6,
    )
    print(f'  [{i+1}] {cap_lp}')


## Bagian 6-Bonus-2 — Batch Inference Benchmark (RNN & LSTM Scratch)

Mengukur **throughput** (gambar/detik) dan **latency** implementasi scratch
RNN dan LSTM pada berbagai ukuran batch.

In [ ]:
from rnn.bonus.bonus_batch_inference import (
    compare_batch_sizes, evaluate_batch_bleu, evaluate_batch_bleu_with_beam
)
from lstm.bonus.bonus_batch_inference import (
    compare_batch_sizes_lstm, evaluate_batch_bleu_lstm,
    evaluate_batch_bleu_lstm_with_beam
)

N_BENCH2 = min(200, len(cnn_test))
BENCH_BATCH_SIZES = (1, 8, 16, 32, 64)

# ── RNN Benchmark ──────────────────────────────────────────────────────────────
print('[Bonus-Bench] RNN Scratch — batch size benchmark:')
rnn_bench = compare_batch_sizes(
    model=rnn_scratch,
    cnn_features=cnn_test[:N_BENCH2],
    gt_captions=gt_captions_test[:N_BENCH2],
    idx2word=idx2word,
    batch_sizes=BENCH_BATCH_SIZES,
    max_length=SEQ_LENGTH,
    n_samples=N_BENCH2,
    verbose=True,
)

# RNN BLEU dengan greedy
print('\n[Bonus-Bench] RNN Scratch BLEU (greedy, test subset):')
rnn_scratch_bleu = evaluate_batch_bleu(
    model=rnn_scratch,
    cnn_features_test=cnn_test[:N_BENCH2],
    gt_captions_test=gt_captions_test[:N_BENCH2],
    idx2word=idx2word,
    max_length=SEQ_LENGTH,
    batch_size=32,
    verbose=True,
)

# RNN BLEU dengan beam search
print('\n[Bonus-Bench] RNN Scratch BLEU (beam search k=5, test subset):')
rnn_scratch_bleu_beam = evaluate_batch_bleu_with_beam(
    model=rnn_scratch,
    cnn_features_test=cnn_test[:N_BENCH2],
    gt_captions_test=gt_captions_test[:N_BENCH2],
    idx2word=idx2word,
    max_length=SEQ_LENGTH,
    k=5,
    batch_size=32,
    verbose=True,
)

# ── LSTM Benchmark ─────────────────────────────────────────────────────────────
print('\n[Bonus-Bench] LSTM Scratch — batch size benchmark:')
lstm_bench = compare_batch_sizes_lstm(
    model=lstm_scratch,
    cnn_features=cnn_test[:N_BENCH2],
    gt_captions=gt_captions_test[:N_BENCH2],
    idx2word=idx2word,
    batch_sizes=BENCH_BATCH_SIZES,
    max_length=SEQ_LENGTH,
    n_samples=N_BENCH2,
    verbose=True,
)

# LSTM BLEU dengan greedy & beam
print('\n[Bonus-Bench] LSTM Scratch BLEU (greedy, test subset):')
lstm_scratch_bleu = evaluate_batch_bleu_lstm(
    model=lstm_scratch,
    cnn_features_test=cnn_test[:N_BENCH2],
    gt_captions_test=gt_captions_test[:N_BENCH2],
    idx2word=idx2word,
    max_length=SEQ_LENGTH,
    batch_size=32,
    verbose=True,
)

print('\n[Bonus-Bench] LSTM Scratch BLEU (beam search k=5, test subset):')
lstm_scratch_bleu_beam = evaluate_batch_bleu_lstm_with_beam(
    model=lstm_scratch,
    cnn_features_test=cnn_test[:N_BENCH2],
    gt_captions_test=gt_captions_test[:N_BENCH2],
    idx2word=idx2word,
    max_length=SEQ_LENGTH,
    k=5,
    batch_size=32,
    verbose=True,
)


## Bagian 7 (Bonus) — Training RNN & LSTM From Scratch (NumPy BPTT)

Demonstrasi **Backpropagation Through Time (BPTT)** secara penuh menggunakan
implementasi NumPy (tanpa Keras):
1. `gradient_checker_rnn / lstm` — verifikasi gradien analitik vs numerik
2. `train_rnn_scratch / train_lstm_scratch` — training loop lengkap dengan Adam

> **Dataset**: subset kecil (200 training, 50 val) agar dapat dijalankan di notebook.
> Untuk training penuh, tingkatkan ukuran data dan jumlah epoch.

In [ ]:
from rnn.bonus.bonus_backward import (
    gradient_checker_rnn, train_one_step, train_rnn_scratch
)
from lstm.bonus.bonus_backward import (
    gradient_checker_lstm, train_one_step_lstm, train_lstm_scratch
)
from rnn.scratch.model_scratch import RNNScratch as _RNNScratch2
from lstm.scratch.model_scratch import LSTMScratch as _LSTMScratch2
import numpy as np

# ── Data subset kecil ─────────────────────────────────────────────────────────
N_SMALL = 200
N_VAL   = 50
X_small     = cnn_train[:N_SMALL]
seq_small   = seq_train[:N_SMALL]
lbl_small   = lbl_train[:N_SMALL]
X_val_sm    = cnn_val[:N_VAL]
seq_val_sm  = seq_val[:N_VAL]
lbl_val_sm  = lbl_val[:N_VAL]

# ── RNN Gradient Checker ───────────────────────────────────────────────────────
print('[Bonus-BPTT] RNN Gradient Checker (sample tunggal, eps=1e-4)...')
print('  Verifikasi: gradien analitik (backward) vs numerik (finite diff).')

_rnn_tiny = _RNNScratch2(
    vocab_size=VOCAB_SIZE, embed_dim=64, hidden_dim=64,
    num_layers=1, feature_dim=FEATURE_DIM
)
_rnn_tiny.build()

gradient_checker_rnn(
    model=_rnn_tiny,
    cnn_feature_sample=X_small[:1],
    token_sample=seq_small[:1],
    label_sample=lbl_small[:1],
    epsilon=1e-4,
    verbose=True,
)

# ── RNN Training from Scratch ─────────────────────────────────────────────────
print('\n[Bonus-BPTT] Training RNN from scratch (3 epoch, Adam, subset 200 sampel)...')
rnn_scratch_trained = train_rnn_scratch(
    model=_rnn_tiny,
    train_features=X_small,
    train_seqs=seq_small,
    train_targets=lbl_small,
    val_features=X_val_sm,
    val_seqs=seq_val_sm,
    val_targets=lbl_val_sm,
    epochs=3,
    batch_size=16,
    lr=0.001,
    optimizer='adam',
    verbose=True,
)

# ── LSTM Gradient Checker ─────────────────────────────────────────────────────
print('\n[Bonus-BPTT] LSTM Gradient Checker (sample tunggal, eps=1e-4)...')

_lstm_tiny = _LSTMScratch2(
    vocab_size=VOCAB_SIZE, embed_dim=64, hidden_dim=64,
    num_layers=1, feature_dim=FEATURE_DIM
)
_lstm_tiny.build()

gradient_checker_lstm(
    model=_lstm_tiny,
    cnn_feature_sample=X_small[:1],
    token_sample=seq_small[:1],
    label_sample=lbl_small[:1],
    epsilon=1e-4,
    verbose=True,
)

# ── LSTM Training from Scratch ────────────────────────────────────────────────
print('\n[Bonus-BPTT] Training LSTM from scratch (3 epoch, Adam, subset 200 sampel)...')
lstm_scratch_trained = train_lstm_scratch(
    model=_lstm_tiny,
    train_features=X_small,
    train_seqs=seq_small,
    train_targets=lbl_small,
    val_features=X_val_sm,
    val_seqs=seq_val_sm,
    val_targets=lbl_val_sm,
    epochs=3,
    batch_size=16,
    lr=0.001,
    optimizer='adam',
    verbose=True,
)
print('[Bonus-BPTT] Selesai.')
